# پیوت تسویه — از PDF فارسی تا پژوهش تیک‌به‌تیک

فقط منطق PDF «ستاپ پیوت تسویه جلیل ضرغام» بررسی می‌شود، نه راهبردهای نامرتبط. **AB صعودی → فروش بعد از تریگر معکوس؛ AB نزولی → خرید. شکست B به‌تنهایی ورود نیست.**

انتخاب شما: هر دو برداشت سایه/کلوز، TP ثابت ۲٫۵/۳R، فعلاً تست با داده موجود. ۲۰ تصویر یکتا DAX/FXCM یک‌دقیقه‌ای ساختاری بررسی شدند. فایل‌های با نام 04-26 و 05-05 در تصویر تاریخ 04-23 و 05-06 دارند. یک تصویر تکراری است.

**تطبیق عددی سیگنال تصاویر DAX تأیید نشده**: داده خام همان نماد و ساعت نمودار موجود نیست. نمونه اجرا GBPUSD دارای Bid/Ask است، نه DAX. تصاویر منتخب و R نام فایل‌ها برچسب یادگیری یا مدرک نرخ برد نیستند. هدف ماهانه ۳–۴٪ و DD<۱۰٪ هنوز اثبات نشده است.

نوت‌بوک خودکفا؛ همه سلول‌ها را به ترتیب اجرا کنید. پیش‌فرض sample فقط آزمون مهندسی است. برای تاریخچه خودتان local یا histdata را انتخاب کنید. اکوییتی و هزینه‌ها هر تیک اجرا می‌شوند؛ داده خام چندساله RAM زیادی می‌خواهد.


In [ ]:
# @title ۱ — دریافت تیک واقعی (همان روش token + POST فایل ارسالی)
import sys, subprocess, importlib.util, json, glob, hashlib
from pathlib import Path
requirements = ['numpy>=2.2,<2.5', 'pandas>=2.2,<3.1', 'pyarrow>=18,<26',
                'numba>=0.61,<0.69', 'scikit-learn>=1.6,<1.10', 'matplotlib>=3.9,<3.12',
                'requests>=2.32,<3', 'beautifulsoup4>=4.12,<5']
modules = ['numpy','pandas','pyarrow','numba','sklearn','matplotlib','requests','bs4']
if any(importlib.util.find_spec(m) is None for m in modules):
    subprocess.check_call([sys.executable,'-m','pip','install',*requirements])
import numpy as np, pandas as pd
from IPython.display import display, HTML
PACKAGE = Path('.hippo_notebook') / 'hippo_lab'
PACKAGE.mkdir(parents=True, exist_ok=True)
(PACKAGE / '__init__.py').write_text('')
sys.path.insert(0, str(PACKAGE.parent.resolve()))
# Re-running all cells reloads the embedded implementation, not stale imports.
for name in list(sys.modules):
    if name == 'hippo_lab' or name.startswith('hippo_lab.'):
        del sys.modules[name]
(PACKAGE / 'data.py').write_text('"""Quote loaders. No price-only fallback and no synthetic quotes."""\nfrom pathlib import Path\nfrom datetime import timezone, timedelta\nimport hashlib, io, zipfile, time\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom bs4 import BeautifulSoup\n\nHISTDATA_TZ = timezone(timedelta(hours=-5))  # fixed EST, NOT America/New_York\nSAMPLE_COMMIT = \'1e0879d8214bafffe44d48e61d1ec6410ec4e7ec\'\n\n\ndef validate_ticks(frame, naive_tz=None):\n    f = frame.rename(columns={c: str(c).lower() for c in frame.columns}).copy()\n    if \'timestamp\' in f:\n        f.index = pd.DatetimeIndex(pd.to_datetime(f.pop(\'timestamp\'), format=\'mixed\'))\n    if not isinstance(f.index, pd.DatetimeIndex):\n        raise ValueError(\'DatetimeIndex or timestamp column required\')\n    if f.index.tz is None:\n        if naive_tz is None:\n            raise ValueError(\'Naive timestamps: explicitly provide source timezone\')\n        f.index = f.index.tz_localize(naive_tz, ambiguous=\'raise\', nonexistent=\'raise\')\n    f.index = f.index.tz_convert(\'UTC\').as_unit(\'ns\')\n    if not {\'bid\', \'ask\'} <= set(f):\n        raise ValueError(\'Real bid AND ask required. Price-only data are not executable quotes.\')\n    f = f[[\'bid\', \'ask\']].astype(float)\n    good = np.isfinite(f).all(axis=1) & (f.bid > 0) & (f.ask >= f.bid)\n    audit = {\'input_rows\': len(f), \'invalid_rows\': int((~good).sum()),\n             \'out_of_order_rows\': int((f.index.to_series().diff().dt.total_seconds() < 0).sum()),\n             \'duplicate_timestamps\': int(f.index.duplicated().sum())}\n    # Preserve differing quotes with equal timestamps, and their original source order.\n    f = f.loc[good].sort_index(kind=\'stable\')\n    if len(f) < 2:\n        raise ValueError(\'Insufficient valid quotes\')\n    gaps = f.index.to_series().diff().dt.total_seconds()\n    audit.update(valid_rows=len(f), start=str(f.index[0]), end=str(f.index[-1]),\n                 gaps_over_60s=int((gaps > 60).sum()), gaps_over_5min=int((gaps > 300).sum()), max_gap_seconds=float(gaps.max()),\n                 median_spread=float((f.ask-f.bid).median()),\n                 p99_spread=float((f.ask-f.bid).quantile(.99)))\n    return f, audit\n\n\ndef read_ticks(path, naive_tz=HISTDATA_TZ):\n    p = Path(path)\n    if p.suffix == \'.parquet\':\n        return validate_ticks(pd.read_parquet(p), naive_tz)\n    if p.suffix == \'.zip\':\n        with zipfile.ZipFile(p) as z:\n            names = [n for n in z.namelist() if n.lower().endswith(\'.csv\')]\n            if len(names) != 1:\n                raise ValueError(\'Choose a ZIP containing exactly one quote CSV\')\n            with z.open(names[0]) as stream:\n                f = pd.read_csv(stream, header=None, names=[\'timestamp\',\'bid\',\'ask\',\'volume\'])\n        f.timestamp = pd.to_datetime(f.timestamp, format=\'%Y%m%d %H%M%S%f\')\n        return validate_ticks(f, naive_tz)\n    return validate_ticks(pd.read_csv(p), naive_tz)\n\n\ndef download_histdata(pair, year, month, folder=\'data/histdata\', retries=3):\n    """Same form-token + POST method as the supplied Colab, fail loudly."""\n    pair = pair.replace(\'/\', \'\').upper()\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    dest = folder / f\'{pair}_{year}_{month:02d}.zip\'\n    if dest.exists():\n        read_ticks(dest)  # never silently trust a partial archive\n        return dest\n    page = f\'https://www.histdata.com/download-free-forex-historical-data/?/ascii/tick-data-quotes/{pair.lower()}/{year}/{month}\'\n    errors = []\n    for attempt in range(retries):\n        try:\n            with requests.Session() as s:\n                s.headers.update({\'User-Agent\':\'Mozilla/5.0\', \'Referer\':page})\n                r = s.get(page, timeout=45); r.raise_for_status()\n                form = BeautifulSoup(r.content, \'html.parser\').find(\'form\', id=\'file_down\')\n                if form is None: raise ValueError(\'HistData token form not found\')\n                payload = {t.get(\'name\'): t.get(\'value\',\'\') for t in form.find_all(\'input\') if t.get(\'name\')}\n                r = s.post(\'https://www.histdata.com/get.php\', data=payload, timeout=120)\n                r.raise_for_status()\n                if not zipfile.is_zipfile(io.BytesIO(r.content)): raise ValueError(\'Response is not a ZIP\')\n                tmp = dest.with_suffix(\'.partial.zip\'); tmp.write_bytes(r.content)\n                read_ticks(tmp); tmp.replace(dest)\n                return dest\n        except (requests.RequestException, ValueError, zipfile.BadZipFile) as e:\n            errors.append(str(e)); time.sleep(2**attempt)\n    raise RuntimeError(f\'Failed {pair} {year}-{month:02d}: {errors}; upload original quote ZIP/parquet instead\')\n\n\ndef download_sample(folder=\'data/cache\'):\n    """A public GAIN Capital mirror: provenance claimed by repo, not broker-certified."""\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    p = folder/\'gain_sample.csv\'\n    if not p.exists():\n        url = f\'https://codeload.github.com/phuocidi/cleaned_tick_data/zip/{SAMPLE_COMMIT}\'\n        r = requests.get(url, timeout=90); r.raise_for_status()\n        with zipfile.ZipFile(io.BytesIO(r.content)) as z:\n            n = next(n for n in z.namelist() if n.endswith(\'2016_04_GBP_USD_Week1.csv\'))\n            p.write_bytes(z.read(n))\n    f = pd.read_csv(p, header=None, names=[\'pair\',\'timestamp\',\'bid\',\'ask\'])\n    # GAIN historical time is New York local (DST applies); verify against broker before deployment.\n    ticks, audit = validate_ticks(f, \'America/New_York\')\n    audit.update(source=\'Public mirror of GAIN Capital GBPUSD Week1 April 2016\',\n                 commit=SAMPLE_COMMIT, sha256=hashlib.sha256(p.read_bytes()).hexdigest(),\n                 timezone_assumption=\'America/New_York; not independently broker-certified\')\n    return ticks, audit\n\n\ndef make_bars(ticks, rule=\'15min\'):\n    mid = (ticks.bid + ticks.ask)/2\n    bars = mid.resample(rule, label=\'right\', closed=\'left\').ohlc()\n    bars[\'count\'] = mid.resample(rule, label=\'right\', closed=\'left\').count()\n    bars[\'spread\'] = (ticks.ask-ticks.bid).resample(rule, label=\'right\', closed=\'left\').median()\n    bars = bars.dropna()\n    # Do not treat the final unfinished candle as closed.\n    return bars.loc[bars.index <= ticks.index[-1]]\n', encoding='utf-8')

from hippo_lab.data import download_histdata, download_sample, read_ticks, make_bars, validate_ticks, HISTDATA_TZ
DATA_MODE = 'sample' # @param ['sample', 'histdata', 'local']
PAIR = 'GBPUSD' # @param {type:'string'}
ACCOUNT_CURRENCY = 'USD' # @param {type:'string'}
START_YEAR = 2022 # @param {type:'integer'}
END_YEAR = 2024 # @param {type:'integer'}
LOCAL_GLOB = '/content/drive/MyDrive/HIPO_DATA/GBPUSD_TICK_*.parquet' # @param {type:'string'}
MOUNT_DRIVE = False # @param {type:'boolean'}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if DATA_MODE == 'sample':
    if PAIR != 'GBPUSD': raise ValueError('The pinned sample is GBPUSD only')
    ticks, data_audit = download_sample()
else:
    files = []
    if DATA_MODE == 'histdata':
        now = pd.Timestamp.now(tz='UTC')
        for year in range(START_YEAR, END_YEAR+1):
            for month in range(1,13):
                if pd.Timestamp(year=year,month=month,day=1,tz='UTC')+pd.DateOffset(months=1)>now:
                    continue # skip incomplete current month
                print(f'Downloading {PAIR} {year}-{month:02d}', flush=True)
                files.append(download_histdata(PAIR,year,month))
    elif DATA_MODE == 'local':
        files = sorted(Path(p) for p in glob.glob(LOCAL_GLOB))
    if not files: raise ValueError('No quote files. Mount Drive or correct LOCAL_GLOB.')
    frames=[]; audits=[]; manifest=[]
    for path in files:
        f,a=read_ticks(path, naive_tz=HISTDATA_TZ)
        # Annual files from original downloader use Bid/Ask and fixed EST naive index.
        frames.append(f); audits.append(a)
        h=hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
        manifest.append({'file':str(path),'sha256':h.hexdigest()})
    ticks, data_audit=validate_ticks(pd.concat(frames))
    del frames
    # Overlapping files are dangerous: equal timestamps with different quotes may be legitimate
    # inside one file, but cross-file overlaps must be corrected by the user.
    ranges=sorted((pd.Timestamp(a['start']),pd.Timestamp(a['end'])) for a in audits)
    if any(b[0]<=a[1] for a,b in zip(ranges,ranges[1:])):
        raise ValueError('Overlapping quote files; choose non-overlapping monthly OR annual files, not both')
    data_audit.update(source='HistData quote ZIP / user original parquet',pair=PAIR,
                      file_audits=audits, manifest=manifest, timezone='HistData fixed EST UTC-5')
if len(ticks)>100_000_000:
    print('WARNING: >100M quotes. High-RAM Colab required; keep only one pair. No downsampling of execution quotes.')
# M1 bid bars are created after loading the pivot module
print(json.dumps(data_audit, indent=2, ensure_ascii=False))
print('Quote RAM MB:',round(ticks.memory_usage(deep=True).sum()/1e6,1))
display(ticks.head())


# منطق پیوت تسویه — تفکیک متن، برداشت و موارد تأییدنشده

منبع: PDF «ستاپ پیوت تسویه جلیل ضرغام»، دو صفحه، پوشه Doc در commit `9ae9cdb6ad4b4469fec89fdd8280f6ed011b313c`. همه ۲۰ تصویر یکتا بررسی شدند؛ یک PNG تکراری است. انتخاب کاربر: هر دو برداشت سایه/کلوز، TP ثابت ۲٫۵/۳R، آزمون فعلاً با داده موجود، نه انتظار برای داده DAX.

## جهت معامله
AB صعودی => ستاپ **فروش** بعد از شکست رو به پایین کف سیگنال. AB نزولی => نسخه قرینه **خرید** بعد از شکست رو به بالای سقف سیگنال. شکست B خود ورود نیست. C نباید از A عبور کند؛ AB>CD اجباری است. D تا تریگر ممکن است بزرگ‌تر شود ولی محدوده مجاز نباید نقض شود.

## قواعد صریح PDF
- AB شارپ، معمولاً حداقل سه کندل و حداکثر یک نویز. دو کندل با رنج هرکدام بیش از ATR با تخفیف مجاز. AB بلندتر، دامنه حدود ۲–۵ ATR.
- BC حداقل سه کندل، اصلاح ۲۰–۵۰٪ AB، بسته‌شدن فراتر از ۵۰٪ ممنوع، لمس A حتی با سایه ابطال. BC می‌تواند بیس یا روند باشد.
- واگرایی زمانی BC>AB اختیاری است، نه الزام ورود.
- شکست B باید با بدنه/ATR مناسب و سایه بیرونی کوچک باشد، اکسترمم تازه نسبت به کندل‌های بین B و شکست بسازد. حدوداً B وسط رنج کندل باشد.
- FL از دو/سه کندل قابل تجمیع است، کندل دوم/سوم باید اکسترمم اول را بشکند؛ کندل ترکیبی دوباره معیار کیفیت را بگذراند.
- بین سیگنال و تأیید فاصله ممکن است وجود داشته باشد؛ نقض AB>CD مجاز نیست.
- تأیید با شکست بدنه‌ای کف/سقف سیگنال و بدنه با ATR خوب. استاپ پشت سیگنال یا اکسترمم مینوری تا تأیید. TP حدود ۲٫۵/۳R.

## تبدیل عملیاتی (فرض‌های قابل تغییر، نه نقل قول PDF)
- کندل‌های یک‌دقیقه‌ای **Bid**، برچسب زمان پایان؛ ATR وایلدر ۱۴، تنها تا کندل قبلی.
- AB در اولین کندل اصلاحی که B را توسعه نمی‌دهد قفل می‌شود؛ از موج‌های پایان‌یافته طول ۲ تا ۱۲، بلندترین موج واجد شرایط انتخاب می‌شود. بدون fractal آینده. این segmentation ممکن است با چشم معامله‌گر تفاوت داشته باشد.
- نویز = کندل غیرهم‌جهت؛ نسبت کارایی `(close_B-open_A)/sum(range)` >=۰٫۵۵، میانه رنج/ATR>=۰٫۶. A و B اکسترمم واقعی ابتدای/انتهای موج‌اند.
- C تا اولین شکست B موقت است؛ در آن لحظه نهایی/قفل می‌شود. تعداد BC از بعد B تا C نهایی است، **نه همه بیس**؛ کندل‌های بیس پس از C جدا گزارش می‌شوند. شکست B قبل از رسیدن C به حداقل سه کندل رد می‌شود.
- سیگنال نسبت بدنه به رنج>=۰٫۶، نسبت سایه بیرونی<=۰٫۲۵، رنج>=۰٫۸ATR، جای B در ۲۰–۸۰٪ رنج. این اعداد عملیاتی‌اند و نیاز به تطبیق داده/نظر معامله‌گر دارند.
- تریگر بازشونده در سمت داخل سطح و بسته‌شونده در سمت شکسته‌شده، بدنه>=۰٫۵ATR. گپ کامل روی سطح به عنوان شکست بدنه‌ای قبول نمی‌شود.
- استاپ = اکسترمم مشاهده‌شده از B تا تأیید + بافر ۰٫۱ATR؛ استاپ **قیمت ثابت** است، با ورود تیک بعد به‌صورت شناور جابه‌جا نمی‌شود. TP از قیمت واقعی ورود با RR ثابت ساخته می‌شود؛ RR خالص به دلیل هزینه‌ها کمتر است.
- انقضای بیس/انتظار تریگر هرکدام ۱۲۰ کندل؛ PDF عددی ندارد. اختلاف طول زمان >۳ برابر فاصله معمول، state را ریست می‌کند. فقط یک state در هر جهت فعال است؛ ستاپ‌های هم‌پوشان دیگری ممکن است حذف شوند، در گزارش محدودیت است.

## دو برداشت درخواست‌شده
**wick:** اصلاح سایه BC هم <=۵۰٪ و اکسترمم D باید اکیداً داخل C±AB باشد. **close:** سایه BC می‌تواند از ۵۰٪ عبور کند تا A را لمس نکند؛ کلوز فراتر از ۵۰٪ ممنوع. سایه D می‌تواند از باکس عبور کند، اما کلوز >=حد باکس ابطال است. هیچ‌یک تطبیق قطعی با اجرای دستی تأیید نشده‌اند.

## مواردی که تصاویر تعیین نمی‌کنند
اعداد ATR دقیق، منبع ساعت، قیمت OHLC، زمان دقیق سیگنال در برخی موارد، خروج واقعی و trailing/partial exits. نام‌های ۱۴٫۶R و مانند آن برچسب یادگیری یا خروج الگوریتم نیستند. خروج ثابت انتخاب کاربر است. تصاویر منتخب، نرخ برد کل سیستم را تعیین نمی‌کنند.

## شرط تطبیق واقعی
همان نماد FXCM_GER30، داده همان تاریخ‌ها، منطقه زمانی TradingView و تعریف timestamp لازم است. اکنون فقط تطبیق **ساختاری دستی** و نمودار خودکار **GBPUSD** ارائه شده؛ `exact_DAX_image_signal_match_verified=false`. تفاوت تاریخ داخل دو نمودار در image_review.csv ثبت است. نمی‌توان ادعا کرد کد همان سیگنال‌های تصویر را داده است.

## ML برای کاهش خطا، بدون پنهان‌کردن سود ازدست‌رفته
مدل فقط meta-filter روی رویداد همین منطق است؛ ویژگی whitelist مربوط به AB/BC/CD/FL/تریگر و ریسک. برچسب نتیجه تیک واقعی با استاپ ثابت و TP ثابت است. آموزش با پنجره‌های exposure غیرهم‌پوشان، پایان label قبل از مرز و embargo زمانی ۲۴ساعته. حداقل ۱۰۰ برچسب آموزش و حداقل ۳۰ برد و ۳۰ باخت اعتبارسنجی. آستانه تنها روی validation، حفظ >=۹۵٪ تعداد و پول بردها، حذف >=۱۰٪ باخت‌ها و بهبود PnL. هیچ تضمینی برای حذف فقط باخت‌ها نیست؛ بردهای حذف‌شده و سود ازدست‌رفته در OOS صریح گزارش می‌شوند. بدون شواهد کافی فیلتر bypass است، نه ادعای موفقیت ML.


In [ ]:
# @title ۲ — موتور اجرا، قواعد پیوت و ابزار تطبیق
(PACKAGE / 'engine.py').write_text('"""Single-position event-driven quote replay; account currency = pair quote currency."""\nfrom dataclasses import dataclass, asdict\nimport numpy as np\nimport pandas as pd\nfrom numba import njit\n\n@dataclass(frozen=True)\nclass Execution:\n    pair: str = "GBPUSD"\n    account_currency: str = "USD"\n    initial_cash: float = 10000.0\n    risk_fraction: float = .0025\n    max_leverage: float = 5.0\n    slippage: float = .00002  # absolute quote-price units per fill\n    commission_per_million: float = 35.0  # per SIDE, quote currency / million BASE units\n    spread_multiplier: float = 1.0\n    tp_trade_through: float = .00001  # require executable quote beyond limit\n    max_spread: float = .00030\n    latency_ms: int = 250\n    max_entry_delay_s: int = 60\n    max_hold_hours: float = 6.0\n    drawdown_breaker: float = .08\n    daily_loss_limit: float = .02\n    session_start_hour: int = 7\n    session_end_hour: int = 16\n    flatten_hour: float = 16.75  # NY, before rollover; no overnight swap model\n\n@njit\ndef _replay(ts, bid0, ask0, local_day, hour, signal_tick, sides, distances, rr, absolute_stops,\n            cash0, risk, leverage, slip, fee, spread_mult, max_spread,\n            hold_ns, breaker, daily_limit, tp_through, session_start, session_end, flatten_hour):\n    n = len(ts); eq = np.empty(n); ledger = np.empty((len(sides)+1, 9))\n    cash = cash0; peak = cash0; day_cash = cash0; day = local_day[0]\n    pos = 0; units = 0.; entry = 0.; stop = 0.; target = 0.; entry_i = 0\n    ptr = 0; trades = 0; halted = False; day_halt = False; max_dd = 0.\n    for i in range(n):\n        spread = (ask0[i]-bid0[i])*spread_mult\n        mid = (ask0[i]+bid0[i])/2\n        bid = mid-spread/2; ask = mid+spread/2\n        liquidation = bid if pos == 1 else ask\n        mark = cash + pos*units*(liquidation-entry) - (units*fee if pos else 0.)\n        if local_day[i] != day:\n            day = local_day[i]; day_cash = mark; day_halt = False\n        peak = max(peak, mark); dd = max(0., 1-mark/peak); max_dd = max(max_dd, dd)\n        if dd >= breaker: halted = True\n        if mark <= day_cash*(1-daily_limit): day_halt = True\n        exited = False\n        if pos:\n            reason = 0\n            if (pos == 1 and bid <= stop) or (pos == -1 and ask >= stop): reason = 1\n            elif (pos == 1 and bid >= target+tp_through) or (pos == -1 and ask <= target-tp_through): reason = 2\n            elif ts[i]-ts[entry_i] >= hold_ns: reason = 3\n            elif hour[i] >= flatten_hour or local_day[i] != local_day[entry_i]: reason = 4\n            if halted or day_halt: reason = 5\n            if i == n-1: reason = 6\n            if reason:\n                # Stop gaps filled at the observed executable quote, not at requested stop.\n                # Take-profit capped at its limit; no favourable price improvement assumed.\n                fill = liquidation - pos*slip\n                if reason == 2: fill = min(fill, target) if pos == 1 else max(fill, target)\n                pnl = pos*units*(fill-entry) - 2*units*fee\n                cash += pos*units*(fill-entry) - units*fee\n                ledger[trades] = np.array([entry_i,i,pos,units,entry,fill,pnl,reason,stop])\n                trades += 1; pos = 0; exited = True; mark = cash\n        # Signals arriving while busy are discarded, NOT queued for a later entry.\n        while ptr < len(sides) and signal_tick[ptr] <= i:\n            if signal_tick[ptr] == i and not pos and not exited and not halted and not day_halt and i < n-1:\n                d = distances[ptr]; side = sides[ptr]\n                if np.isfinite(absolute_stops[ptr]):\n                    quoted_entry=(ask if side==1 else bid)+side*slip\n                    d=side*(quoted_entry-absolute_stops[ptr])\n                if d > spread+2*slip and spread <= max_spread and session_start <= hour[i] < min(session_end,flatten_hour):\n                    price = (ask if side == 1 else bid) + side*slip\n                    # Include expected stop exit slippage and both commissions in risk budget.\n                    qty = min(cash*risk/(d+slip+2*fee), cash*leverage/price)\n                    if qty > 0:\n                        pos = side; units = qty; entry = price; entry_i = i\n                        stop = absolute_stops[ptr] if np.isfinite(absolute_stops[ptr]) else price-side*d; target = price+side*d*rr[ptr]\n                        cash -= units*fee\n                        mark = cash + pos*units*((bid if pos == 1 else ask)-entry)-units*fee\n            ptr += 1\n        eq[i] = mark\n        peak = max(peak, mark); max_dd = max(max_dd, max(0., 1-mark/peak))\n    return eq, ledger[:trades], max_dd, halted\n\n\ndef backtest(ticks, signals, cfg=Execution()):\n    if len(cfg.pair) != 6 or cfg.account_currency != cfg.pair[-3:]:\n        raise ValueError(\'Account currency must equal quote currency; cross-currency conversion is not implemented\')\n    if cfg.initial_cash <= 0 or not 0 < cfg.risk_fraction < 1 or cfg.slippage < 0 or cfg.spread_multiplier < 1:\n        raise ValueError(\'Invalid execution configuration\')\n    if not ticks.index.is_monotonic_increasing or ticks.index.tz is None:\n        raise ValueError(\'Sorted timezone-aware quotes required\')\n    if (ticks.ask < ticks.bid).any(): raise ValueError(\'Crossed quotes\')\n    if len(ticks) < 2: raise ValueError(\'At least two quotes required\')\n    ts = ticks.index.as_unit(\'ns\').asi8\n    s = signals.sort_index(kind=\'stable\')\n    if len(s) and (not s.index.is_unique or not s.side.isin([-1,1]).all() or (s.distance <= 0).any() or (s.rr <= 0).any()):\n        raise ValueError(\'Signals must have unique times, side +/-1 and positive distance/rr\')\n    ready = s.index.as_unit(\'ns\').asi8 + cfg.latency_ms*1_000_000\n    # Strictly later timestamp even at zero configured latency.\n    inds = np.maximum(np.searchsorted(ts, ready, side=\'left\'), np.searchsorted(ts, s.index.as_unit(\'ns\').asi8, side=\'right\'))\n    valid = inds < len(ts)\n    safe = np.minimum(inds, len(ts)-1)\n    valid &= ts[safe]-ready <= cfg.max_entry_delay_s*1_000_000_000\n    local = ticks.index.tz_convert(\'America/New_York\')\n    days = np.asarray(local.year*10000+local.month*100+local.day, dtype=np.int64)\n    hours = np.asarray(local.hour+local.minute/60+local.second/3600, dtype=float)\n    equity, ledger, max_dd, halted = _replay(ts, ticks.bid.to_numpy(), ticks.ask.to_numpy(), days, hours,\n        inds[valid], s.side.to_numpy(dtype=np.int64)[valid], s.distance.to_numpy(dtype=float)[valid], s.rr.to_numpy(dtype=float)[valid],\n        s.stop_price.to_numpy(dtype=float)[valid] if \'stop_price\' in s else np.full(int(valid.sum()),np.nan),\n        cfg.initial_cash, cfg.risk_fraction, cfg.max_leverage, cfg.slippage, cfg.commission_per_million/1e6,\n        cfg.spread_multiplier, cfg.max_spread, int(cfg.max_hold_hours*3600e9), cfg.drawdown_breaker,\n        cfg.daily_loss_limit, cfg.tp_trade_through, cfg.session_start_hour, cfg.session_end_hour, cfg.flatten_hour)\n    trades = pd.DataFrame(ledger, columns=[\'entry_tick\',\'exit_tick\',\'side\',\'units\',\'entry\',\'exit\',\'pnl\',\'reason\',\'stop\'])\n    if len(trades):\n        gap_seconds=np.r_[0.,np.diff(ts)/1e9]\n        gap_prefix=np.cumsum(gap_seconds>60)\n        trades[\'quote_gap_over_60s\']=gap_prefix[trades.exit_tick.astype(int)]>gap_prefix[trades.entry_tick.astype(int)]\n        trades[\'planned_loss\'] = trades.units*((trades.entry-trades.stop).abs()+cfg.slippage+2*cfg.commission_per_million/1e6)\n        trades[\'realized_R\'] = trades.pnl/trades.planned_loss\n        trades[\'entry_time\'] = ticks.index[trades.entry_tick.astype(int)]\n        trades[\'exit_time\'] = ticks.index[trades.exit_tick.astype(int)]\n    return {\'equity\':pd.Series(equity,index=ticks.index,name=\'equity\'), \'trades\':trades,\n            \'max_drawdown\':float(max_dd), \'halted\':bool(halted), \'config\':asdict(cfg),\n            \'submitted_signals\':len(s), \'stale_or_outside_signals\':int((~valid).sum())}\n', encoding='utf-8')
(PACKAGE / 'research.py').write_text('"""Predeclared hypotheses, time-purged ML and nested chronological selection."""\nfrom dataclasses import replace\nimport hashlib, json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom .engine import Execution, backtest\n\nCANDIDATES = [\n    {\'name\':\'breakout_32\', \'family\':\'breakout\', \'lookback\':32, \'stop_atr\':2., \'rr\':1.5},\n    {\'name\':\'breakout_64\', \'family\':\'breakout\', \'lookback\':64, \'stop_atr\':2.5, \'rr\':2.},\n    {\'name\':\'reversion_32\', \'family\':\'reversion\', \'lookback\':32, \'stop_atr\':2., \'rr\':1.},\n    {\'name\':\'reversion_64\', \'family\':\'reversion\', \'lookback\':64, \'stop_atr\':2.5, \'rr\':1.},\n    {\'name\':\'logistic_060\', \'family\':\'ml\', \'threshold\':.60, \'stop_atr\':2., \'rr\':1.5},\n    {\'name\':\'logistic_065\', \'family\':\'ml\', \'threshold\':.65, \'stop_atr\':2., \'rr\':1.5},\n]\nFEATURES = [\'ret1\',\'ret4\',\'ret16\',\'z32\',\'trend\',\'vol\',\'cost\']\nLABEL_HORIZON = 8  # 2 hours on M15\nWARMUP_BARS = 96\nPROTOCOL = {\'candidate_trials\':len(CANDIDATES), \'bar_rule\':\'15min\', \'label_horizon\':LABEL_HORIZON,\n    \'minimum_validation_trades\':30, \'minimum_ml_independent_labels\':100,\n    \'minimum_months_for_target\':24, \'embargo_hours\':24, \'minimum_forward_folds\':4, \'holdout_months\':6, \'selection\':\'positive net return, DD < 8%, maximize net return / max(DD, 0.5%)\',\n    \'no_passing_candidate\':\'CASH\', \'seed\':8056}\n\n\ndef protocol_hash(cfg):\n    from dataclasses import asdict\n    return hashlib.sha256(json.dumps({\'protocol\':PROTOCOL, \'candidates\':CANDIDATES,\n                                      \'execution\':asdict(cfg)},sort_keys=True).encode()).hexdigest()\n\n\ndef features(b):\n    c = b.close; out = pd.DataFrame(index=b.index)\n    tr = pd.concat([b.high-b.low,(b.high-c.shift()).abs(),(b.low-c.shift()).abs()],axis=1).max(axis=1)\n    out[\'atr\'] = tr.rolling(32,min_periods=32).mean()\n    for k in [1,4,16]: out[f\'ret{k}\'] = c.pct_change(k, fill_method=None)\n    out[\'z32\'] = (c-c.rolling(32).mean())/c.rolling(32).std()\n    out[\'trend\'] = (c.rolling(16).mean()-c.rolling(64).mean())/out.atr\n    out[\'vol\'] = out.ret1.rolling(32).std()\n    out[\'cost\'] = b.spread/out.atr\n    return out.replace([np.inf,-np.inf],np.nan)\n\n\ndef fit_candidate(candidate, bars, train_start, train_end):\n    if candidate[\'family\'] != \'ml\': return None\n    x = features(bars)\n    future = bars.close.shift(-LABEL_HORIZON)/bars.close-1\n    label_end = pd.Series(bars.index,index=bars.index).shift(-LABEL_HORIZON)\n    # Earliest feature windows entirely inside train; labels end strictly before boundary.\n    cutoff=train_end-pd.Timedelta(hours=PROTOCOL[\'embargo_hours\'])\n    train = (bars.index >= train_start) & (bars.index < cutoff)\n    eligible = train & (label_end < cutoff) & x[FEATURES].notna().all(axis=1)\n    eligible[:np.searchsorted(bars.index, train_start)+WARMUP_BARS] = False\n    loc = np.flatnonzero(eligible)\n    # Non-overlapping labels for training; effective sample count logged/guarded.\n    loc = loc[::LABEL_HORIZON]\n    if len(loc) < PROTOCOL[\'minimum_ml_independent_labels\'] or (future.iloc[loc] > 0).nunique() < 2:\n        return \'INSUFFICIENT\'\n    model = make_pipeline(StandardScaler(), LogisticRegression(C=.1, max_iter=500, random_state=8056))\n    model.fit(x[FEATURES].iloc[loc], (future.iloc[loc]>0).astype(int))\n    return model\n\n\ndef signals_for(candidate, bars, start, end, model=None, embargo=True):\n    x = features(bars); side = pd.Series(0,index=bars.index,dtype=int)\n    if candidate[\'family\'] == \'breakout\':\n        n = candidate[\'lookback\']\n        side.loc[bars.close > bars.high.rolling(n).max().shift(1)] = 1\n        side.loc[bars.close < bars.low.rolling(n).min().shift(1)] = -1\n    elif candidate[\'family\'] == \'reversion\':\n        n = candidate[\'lookback\']; z = (bars.close-bars.close.rolling(n).mean())/bars.close.rolling(n).std()\n        quiet = x.trend.abs() < .75\n        side.loc[(z < -2) & quiet] = 1; side.loc[(z > 2) & quiet] = -1\n    elif model is not None and not isinstance(model,str):\n        good = x[FEATURES].notna().all(axis=1)\n        prob = pd.Series(np.nan,index=bars.index)\n        prob.loc[good] = model.predict_proba(x.loc[good,FEATURES])[:,1]\n        threshold = candidate[\'threshold\']\n        side.loc[prob > threshold] = 1; side.loc[prob < 1-threshold] = -1\n    # One fresh setup; no repeated re-entry into the same persistent signal.\n    side = side.where(side.ne(side.shift()),0)\n    s = pd.DataFrame({\'side\':side,\'distance\':x.atr*candidate[\'stop_atr\'],\'rr\':candidate[\'rr\']})\n    allowed = (s.index >= start) & (s.index < end) & (s.side != 0) & (s.distance > 0)\n    # Historical bars warm up causal features; no artificial removal of first test days.\n    allowed[:WARMUP_BARS] = False\n    return s.loc[allowed].dropna()\n\n\ndef empty_signals():\n    return pd.DataFrame(columns=[\'side\',\'distance\',\'rr\'], index=pd.DatetimeIndex([],tz=\'UTC\'))\n\n\ndef select(bars,ticks,train_start,val_start,val_end,cfg):\n    rows=[]; best=None; best_score=-np.inf\n    validation = ticks.loc[(ticks.index >= val_start) & (ticks.index < val_end)]\n    if len(validation)<2: raise ValueError(\'Empty validation period\')\n    for c in CANDIDATES:\n        model=fit_candidate(c,bars,train_start,val_start)\n        s=signals_for(c,bars,val_start,val_end,model)\n        r=backtest(validation,s,cfg)\n        ret=r[\'equity\'].iloc[-1]/cfg.initial_cash-1; dd=r[\'max_drawdown\']; count=len(r[\'trades\'])\n        eligible=count>=PROTOCOL[\'minimum_validation_trades\'] and ret>0 and dd<.08\n        score=ret/max(dd,.005) if eligible else -np.inf\n        rows.append({\'candidate\':c[\'name\'],\'return\':ret,\'drawdown\':dd,\'trades\':count,\n                     \'eligible\':eligible,\'ml_fit\': \'insufficient\' if isinstance(model,str) else (\'trained\' if model is not None else \'n/a\')})\n        if score>best_score: best_score=score; best=c\n    return best,pd.DataFrame(rows)\n\n\ndef research_run(ticks,bars,cfg=Execution(),mode=\'sample\'):\n    """Sample = smoke test. Full = 12m train/3m validation/3m OOS + final 6m."""\n    start=ticks.index[0]; end=ticks.index[-1]; selections=[]; combined=[]\n    if mode==\'sample\':\n        # Fixed session boundaries for the pinned sample, not tuned to returns.\n        train_start=pd.Timestamp(\'2016-04-04\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        val_start=pd.Timestamp(\'2016-04-06\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        hold_start=pd.Timestamp(\'2016-04-07\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        chosen,table=select(bars,ticks,train_start,val_start,hold_start,cfg)\n        selections.append(table.assign(fold=\'smoke_validation\'))\n        oos_start=hold_start\n    else:\n        # First complete UTC month, exclude partial starting month.\n        month_start=start.normalize().replace(day=1)\n        if start>month_start: month_start += pd.DateOffset(months=1)\n        month_end=end.normalize().replace(day=1)\n        months=(month_end.year-month_start.year)*12+month_end.month-month_start.month\n        if months<24: raise ValueError(f\'Need >=24 complete calendar months, have {months}. No target claim possible.\')\n        # Refuse obvious missing weekday data, rather than silently treating gaps as flat markets.\n        dates=pd.date_range(month_start,month_end-pd.Timedelta(days=1),freq=\'B\',tz=\'UTC\')\n        observed=ticks.index.normalize().unique()\n        missing=dates.difference(observed)\n        if len(missing)/max(len(dates),1) > .03:\n            raise ValueError(f\'Missing {len(missing)} weekdays; inspect data coverage before research\')\n        ticks=ticks.loc[ticks.index<month_end]\n        end=month_end\n        hold_start=month_end-pd.DateOffset(months=6)\n        oos_start=month_start+pd.DateOffset(months=15)\n        cursor=oos_start\n        while cursor<hold_start:\n            test_end=min(cursor+pd.DateOffset(months=3),hold_start)\n            val_start=cursor-pd.DateOffset(months=3); train_start=val_start-pd.DateOffset(months=12)\n            c,table=select(bars,ticks,train_start,val_start,cursor,cfg)\n            selections.append(table.assign(fold=str(cursor),chosen=c[\'name\'] if c else \'CASH\'))\n            if c:\n                model=fit_candidate(c,bars,train_start,cursor)\n                combined.append(signals_for(c,bars,cursor,test_end,model))\n            cursor=test_end\n        val_start=hold_start-pd.DateOffset(months=3); train_start=val_start-pd.DateOffset(months=12)\n        chosen,table=select(bars,ticks,train_start,val_start,hold_start,cfg)\n        selections.append(table.assign(fold=\'final_selection\',chosen=chosen[\'name\'] if chosen else \'CASH\'))\n    # Freeze candidate before accessing final test outcomes. No holdout-driven retries.\n    model=fit_candidate(chosen,bars,train_start,hold_start) if chosen else None\n    final_signals=signals_for(chosen,bars,hold_start,end,model) if chosen else empty_signals()\n    frozen={\'candidate\':chosen or \'CASH\',\'protocol_hash\':protocol_hash(cfg), \'holdout_start\':str(hold_start)}\n    from pathlib import Path\n    model_state=None\n    if model is not None and not isinstance(model,str):\n        model_state={\'coef\':model[-1].coef_.tolist(),\'intercept\':model[-1].intercept_.tolist(),\n                     \'scaler_mean\':model[0].mean_.tolist(),\'scaler_scale\':model[0].scale_.tolist()}\n    frozen[\'model_state\']=model_state\n    frozen[\'selection_sha256\']=hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest()\n    Path(\'artifacts\').mkdir(exist_ok=True)\n    Path(\'artifacts/frozen_selection.json\').write_text(json.dumps(frozen,indent=2))\n    # Data may already be loaded, but NO holdout outcome feeds selection, refit or this hash.\n    final_ticks=ticks.loc[ticks.index>=hold_start]\n    final=backtest(final_ticks,final_signals,cfg)\n    pre=None\n    if mode!=\'sample\':\n        pre_ticks=ticks.loc[(ticks.index>=oos_start)&(ticks.index<hold_start)]\n        pre_s=pd.concat(combined).sort_index() if combined else empty_signals()\n        pre=backtest(pre_ticks,pre_s,cfg)\n    stresses={}\n    for name,stress_cfg in {\n        \'base\':cfg,\n        \'spread_1.5_slip_2x\':replace(cfg,spread_multiplier=1.5,slippage=cfg.slippage*2),\n        \'spread_2_slip_3x_latency_1s\':replace(cfg,spread_multiplier=2.,slippage=cfg.slippage*3,latency_ms=1000),\n    }.items():\n        stresses[name]=backtest(final_ticks,final_signals,stress_cfg)\n    return {\'mode\':mode,\'selection\':pd.concat(selections,ignore_index=True), \'frozen\':frozen,\n            \'final\':final,\'walk_forward\':pre,\'stress\':stresses,\'final_signals\':final_signals}\n', encoding='utf-8')
(PACKAGE / 'report.py').write_text('"""Auditable metrics; partial months never count toward the monthly objective."""\nfrom pathlib import Path\nimport html, json, base64\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom .research import PROTOCOL\n\n\ndef metrics(result):\n    e=result[\'equity\']; cash=result[\'config\'][\'initial_cash\']; t=result[\'trades\']\n    # Avoid duplicate timestamp reindexing but preserve full quote-level DD from engine.\n    e=e.groupby(level=0).last()\n    daily=e.resample(\'1D\').last().ffill()\n    daily_returns=daily.pct_change(fill_method=None); daily_returns.iloc[0]=daily.iloc[0]/cash-1\n    first=e.index[0]; last=e.index[-1]\n    month_end=e.resample(\'ME\').last()\n    rows=[]; prev=cash\n    for timestamp,value in month_end.items():\n        start=timestamp.normalize().replace(day=1)\n        next_start=start+pd.DateOffset(months=1)\n        # Data must bracket a whole UTC month. Missing weekends are allowed up to 3 days.\n        complete=first<=start+pd.Timedelta(days=3) and last>=next_start-pd.Timedelta(days=3)\n        rows.append({\'month\':str(start.date())[:7], \'return\':value/prev-1, \'complete\':complete})\n        prev=value\n    monthly=pd.DataFrame(rows)\n    pnl=t.pnl if len(t) else pd.Series(dtype=float)\n    wins=pnl[pnl>0].sum(); losses=-pnl[pnl<0].sum()\n    stats={\'total_return\':float(e.iloc[-1]/cash-1), \'max_drawdown\':result[\'max_drawdown\'],\n           \'trades\':len(t),\'win_rate\':float((pnl>0).mean()) if len(t) else None,\n           \'profit_factor\':float(wins/losses) if losses>0 else None,\n           \'net_pnl\':float(e.iloc[-1]-cash), \'mean_trade_pnl\':float(pnl.mean()) if len(t) else None,\n           \'gap_affected_trades\':int(t.quote_gap_over_60s.sum()) if len(t) else 0,\n           \'gap_affected_pnl\':float(t.loc[t.quote_gap_over_60s,\'pnl\'].sum()) if len(t) else 0.,\n           \'worst_realized_R\':float(t.realized_R.min()) if len(t) else None,\n           \'complete_months\':int(monthly.complete.sum()),\'drawdown_breaker_triggered\':result[\'halted\'],\n           \'breaker_overshoot\':max(0.,result[\'max_drawdown\']-result[\'config\'][\'drawdown_breaker\']),\n           \'submitted_signals\':result[\'submitted_signals\'], \'stale_or_outside_signals\':result[\'stale_or_outside_signals\']}\n    full=monthly.loc[monthly.complete,\'return\']\n    stats.update(mean_complete_month_return=float(full.mean()) if len(full) else None,\n                 fraction_months_ge_3pct=float((full>=.03).mean()) if len(full) else None,\n                 worst_month=float(full.min()) if len(full) else None)\n    # Bootstrap contiguous blocks only with enough days, not a week-long smoke test.\n    bootstrap=None\n    if len(daily_returns)>=180:\n        x=daily_returns.to_numpy(); rng=np.random.default_rng(8056); samples=[]; dds=[]\n        for _ in range(1000):\n            starts=rng.integers(0,len(x)-20+1,size=int(np.ceil(len(x)/20)))\n            r=np.concatenate([x[j:j+20] for j in starts])[:len(x)]\n            curve=np.r_[1.,np.cumprod(1+r)]\n            samples.append(curve[-1]**(30/len(r))-1)\n            dds.append(np.max(1-curve/np.maximum.accumulate(curve)))\n        bootstrap={\'monthly_equivalent_mean_ci95\':np.quantile(samples,[.025,.975]).tolist(),\n                   \'dd_p95\':float(np.quantile(dds,.95)),\n                   \'note\':\'Conditional 20-calendar-day block resampling; not a guarantee or selection-bias correction.\'}\n    return stats,monthly,daily_returns,bootstrap\n\n\ndef write_report(run,audit,folder=\'artifacts\',diagnostics=None):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    result=run[\'final\']; stats,monthly,daily,bootstrap=metrics(result)\n    no_selection=run[\'frozen\'][\'candidate\']==\'CASH\'\n    enough=run[\'mode\']!=\'sample\' and stats[\'trades\']>=30 and stats[\'complete_months\']>=5\n    status=\'NO_SELECTION\' if no_selection and run[\'mode\']!=\'sample\' else (\'INSUFFICIENT_EVIDENCE\' if not enough else \'EVALUATED_NOT_GUARANTEED\')\n    # Walk-forward and final holdout are separate accounts, not an invented stitched equity curve.\n    wf_stats=None\n    if run[\'walk_forward\'] is not None:\n        wf_stats=metrics(run[\'walk_forward\'])[0]\n    stresses={k:metrics(v)[0] for k,v in run[\'stress\'].items()}\n    forward_folds=max(0,run[\'selection\'].fold.nunique()-1) if run[\'mode\']!=\'sample\' else 0\n    sufficient_folds=forward_folds>=PROTOCOL[\'minimum_forward_folds\']\n    point_met=bool(enough and not result[\'halted\'] and stats[\'mean_complete_month_return\']>=.03 and stats[\'max_drawdown\']<.10 and all(s[\'total_return\']>0 and not s[\'drawdown_breaker_triggered\'] for s in stresses.values()))\n    summary={\'status\':status, \'point_estimate_met_unconfirmed\':point_met,\n             \'forward_folds\':forward_folds, \'sufficient_forward_folds\':sufficient_folds,\n             \'long_term_target_verified\':False, \'input_sufficient_for_full_protocol\':run[\'mode\']!=\'sample\',\n             \'no_selection\':no_selection, \'holdout_evidence_sufficient_for_initial_review\':enough,\n             \'monthly_goal_definition\':\'Mean net complete calendar-month return >=3%; not a promise of 3% every month. Long-term target requires >=24 OOS months.\',\'objective\':\'3–4% monthly, maximum equity drawdown <10%; not guaranteed\',\n             \'data_audit\':audit,\'frozen_selection\':run[\'frozen\'],\'final_holdout\':stats,\n             \'walk_forward_separate_account\':wf_stats,\'stress_same_frozen_signals\':stresses,\n             \'bootstrap\':bootstrap, \'limitations\':[\n                 \'Public mirror quotes not independently broker-certified; no exchange order book or market impact.\',\n                 \'Quote-currency account only (GBPUSD: USD). Other pairs require their quote-currency account; no cross-FX conversion.\',\n                 \'Single position, intraday NY liquidation; no swaps; observed quote gaps can overshoot risk controls.\',\n                 \'Best-of-six selection is biased; holdout is not used for tuning. Revisiting after results voids its untouched status.\',\n                 \'One week cannot establish profitability or transfer across markets; >=24 input months is an exploratory run, not 24 OOS months.\',\n                 \'Validation reject -> no deployment, not evidence that cash is superior.\']}\n    (folder/\'summary.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False))\n    run[\'selection\'].to_csv(folder/\'validation_all_candidates.csv\',index=False)\n    result[\'trades\'].to_csv(folder/\'holdout_trades.csv\',index=False)\n    monthly.to_csv(folder/\'holdout_months.csv\',index=False)\n    result[\'equity\'].resample(\'1min\').last().to_csv(folder/\'equity_1min.csv\')\n    # Full tick equity remains in memory; optional parquet audit, not committed.\n    result[\'equity\'].to_frame().to_parquet(folder/\'equity_every_tick.parquet\')\n    if diagnostics is not None: diagnostics.to_csv(folder/\'smoke_unselected_candidates.csv\',index=False)\n    fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)\n    plot_e=result[\'equity\'].groupby(level=0).last()\n    axes[0].plot(plot_e.index,plot_e.values); axes[0].set_ylabel(\'Liquidation equity\')\n    peak=plot_e.cummax().clip(lower=result[\'config\'][\'initial_cash\'])\n    axes[1].plot(plot_e.index,100*(1-plot_e/peak)); axes[1].set_ylabel(\'Drawdown %\')\n    fig.suptitle(status+\' — frozen holdout (CASH may reflect inadequate evidence)\')\n    fig.tight_layout(); fig.savefig(folder/\'equity.png\'); plt.close(fig)\n    def table(df): return df.to_html(index=False,escape=True,float_format=lambda x:f\'{x:.6f}\')\n    body=f\'\'\'<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><title>HIPO audit</title>\n    <style>body{{font-family:system-ui;max-width:1100px;margin:32px auto;padding:20px;background:#f5f7fb;color:#17233a}}table{{direction:ltr;border-collapse:collapse;width:100%;background:white}}td,th{{padding:8px;border:1px solid #ccd}}pre{{direction:ltr;text-align:left;white-space:pre-wrap}}img{{width:100%}}</style>\n    <h1>گزارش پژوهش تیک\u200cبه\u200cتیک HIPO</h1><h2>{status}</h2>\n    <p>این گزارش تضمین سود نیست. داده کوتاه فقط آزمون مهندسی است؛ عدم معامله، تأیید استراتژی نیست.</p>\n    <img src="data:image/png;base64,{base64.b64encode((folder/\'equity.png\').read_bytes()).decode()}"><h2>تمام آزمایش\u200cهای اعتبارسنجی</h2>{table(run[\'selection\'])}\n    <h2>ماه\u200cها (ناقص\u200cها خارج از هدف)</h2>{table(monthly)}\n    <h2>اجرای تشخیصی فرضیه\u200cهای از پیش تعیین\u200cشده — نه انتخاب و نه اثبات سود</h2>{table(diagnostics) if diagnostics is not None else \'\'}\n    <h2>فرض\u200cها، داده و تنش هزینه</h2><pre>{html.escape(json.dumps(summary,indent=2,ensure_ascii=False))}</pre></html>\'\'\'\n    (folder/\'report.html\').write_text(body,encoding=\'utf-8\')\n    return summary\n', encoding='utf-8')
(PACKAGE / 'pivot.py').write_text('"""Causal operational interpretation of the two-page Persian pivot settlement PDF.\nThresholds not specified numerically in the PDF are research assumptions, not recovered rules.\n"""\nfrom dataclasses import dataclass, asdict\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\n\n@dataclass(frozen=True)\nclass PivotRules:\n    boundary: str = \'wick\'  # wick or close; A touch ALWAYS tested on wick\n    rr: float = 2.5\n    atr_period: int = 14\n    ab_max_bars: int = 12\n    ab_efficiency: float = .55\n    ab_min_median_range_atr: float = .60\n    max_noise: int = 1\n    retrace_min: float = .20\n    retrace_max: float = .50\n    bc_min_bars: int = 3\n    max_bc_bars: int = 120  # engineering expiry, NOT specified by PDF\n    max_armed_bars: int = 120\n    signal_body_ratio: float = .60\n    signal_range_atr: float = .80\n    signal_wick_ratio: float = .25\n    b_position_low: float = .20\n    b_position_high: float = .80\n    trigger_body_atr: float = .50\n    stop_buffer_atr: float = .10\n    allow_fl: bool = True\n    require_time_divergence: bool = False  # optional according to PDF\n\nFEATURE_COLUMNS = [\'ab_atr\',\'ab_efficiency\',\'ab_noise\',\'ab_bars\',\'bc_bars\',\'base_bars_after_C\',\'retrace\',\n                   \'cd_ab_ratio\',\'time_divergence\',\'signal_body_ratio\',\'signal_wick_ratio\',\n                   \'signal_range_atr\',\'trigger_body_atr\',\'risk_atr\',\'fl_bars\']\nEVENT_COLUMNS = [\'side\',\'distance\',\'rr\',\'stop_price\',\'setup_id\',\'a_time\',\'b_time\',\'c_time\',\n                 \'cd_start_time\',\'signal_time\',\'a\',\'b\',\'c\',\'d\',\'signal_level\',*FEATURE_COLUMNS]\n\n\ndef atr_prior(bars, period=14):\n    # Wilder ATR; first period observations warm up; excludes the current candle.\n    prev=bars.close.shift()\n    tr=pd.concat([bars.high-bars.low,(bars.high-prev).abs(),(bars.low-prev).abs()],axis=1).max(axis=1)\n    return tr.ewm(alpha=1/period,adjust=False,min_periods=period).mean().shift(1)\n\n\ndef pivot_bars(ticks, rule=\'1min\'):\n    # Same displayed price convention as quote charts: bid OHLC, not synthetic spread-adjusted mid.\n    b=ticks.bid.resample(rule,label=\'right\',closed=\'left\').ohlc().dropna()\n    b[\'count\']=ticks.bid.resample(rule,label=\'right\',closed=\'left\').count()\n    return b.loc[b.index<=ticks.index[-1]]\n\n\ndef validate_bars(bars):\n    if not isinstance(bars.index,pd.DatetimeIndex) or bars.index.tz is None or not bars.index.is_unique or not bars.index.is_monotonic_increasing:\n        raise ValueError(\'Closed bars need unique increasing timezone-aware timestamps\')\n    x=bars[[\'open\',\'high\',\'low\',\'close\']].astype(float)\n    if not np.isfinite(x.to_numpy()).all() or (x.high<x[[\'open\',\'close\',\'low\']].max(axis=1)).any() or (x.low>x[[\'open\',\'close\',\'high\']].min(axis=1)).any():\n        raise ValueError(\'Invalid OHLC\')\n    return x\n\n\ndef _find_ab(o,h,l,c,atr,end,rule):\n    """Called only when first pullback bar closes. Longest eligible completed impulse wins."""\n    if not np.isfinite(atr[end]) or atr[end]<=0:return None\n    for n in range(min(rule.ab_max_bars,end+1),1,-1):\n        start=end-n+1; av=atr[end]\n        # A and B are candle extremes, with no right-side future fractal confirmation.\n        a=l[start]; b=h[end]; amp=b-a\n        if amp<=0 or a>np.min(l[start:end+1]) or b<np.max(h[start:end+1]):continue\n        noise=int(np.sum(c[start:end+1]<=o[start:end+1]))\n        if noise>rule.max_noise:continue\n        ranges=h[start:end+1]-l[start:end+1]\n        if n==2:\n            if not np.all(ranges>av) or noise:continue\n        elif n>3 and not 2<=amp/av<=5:continue\n        efficiency=(c[end]-o[start])/max(np.sum(ranges),1e-12)\n        if efficiency<rule.ab_efficiency or np.median(ranges)/av<rule.ab_min_median_range_atr:continue\n        return dict(ai=start,bi=end,a=a,b=b,amp=amp,atr=av,ab_bars=n,\n                    ab_noise=noise,ab_efficiency=efficiency)\n    return None\n\n\ndef detect_pivots(bars,rules=PivotRules()):\n    if len(bars)>1 and ((bars.index.to_series().diff().dropna()<pd.Timedelta(minutes=1)).any()):\n        raise ValueError(\'Pivot implementation requires closed M1 bars, not sub-minute bars\')\n    if rules.boundary not in [\'wick\',\'close\']:raise ValueError(\'boundary = wick or close\')\n    if rules.rr<=0 or rules.bc_min_bars<3:raise ValueError(\'Invalid pivot rules\')\n    x=validate_bars(bars); ats=atr_prior(x,rules.atr_period).to_numpy()\n    index=x.index; values=x.to_numpy(); emitted=[]; trace=[]; counts=Counter()\n    # M1 contract is explicit; cadence must not be inferred from future timestamps.\n    step=pd.Timedelta(minutes=1)\n    for direction in [1,-1]:\n        # Normalize downward AB by price inversion so every test is expressed as upward AB.\n        if direction==1:o,h,l,c=values.T\n        else:\n            o=-values[:,0]; h=-values[:,2]; l=-values[:,1]; c=-values[:,3]\n        state=None; consumed_b=-1\n        for i in range(1,len(x)):\n            if step is not pd.NaT and index[i]-index[i-1]>3*step:\n                if state:counts[\'reset_data_gap\']+=1\n                state=None\n            if state is None:\n                # First pullback must not extend B. Seed at prior completed impulse.\n                if c[i]>=c[i-1] or h[i]>h[i-1] or i-1<=consumed_b:continue\n                ab=_find_ab(o,h,l,c,ats,i-1,rules)\n                if ab is None:continue\n                state={**ab,\'phase\':\'BC\',\'ci\':i,\'c\':l[i], \'bc_bars\':1,\'base_bars\':1,\'bc_start\':i,\n                       \'highest\':h[i-1],\'fl_start\':None}\n                counts[\'ab_candidates\']+=1\n                trace.append({\'time\':str(index[i]),\'direction\':direction,\'stage\':\'AB_FROZEN\',\'b_time\':str(index[i-1])})\n                # Seed bar may itself already invalidate BC; process below.\n            s=state\n            def kill(reason):\n                counts[reason]+=1\n                trace.append({\'time\':str(index[i]),\'direction\':direction,\'stage\':reason,\'b_time\':str(index[s[\'bi\']])})\n            if l[i]<=s[\'a\']:\n                kill(\'invalid_A_wick_touch\');state=None;continue\n            if s[\'phase\']==\'BC\':\n                # Keep C provisional throughout the base. Freeze it at the first B wick break.\n                # BC duration = B exclusive through FINAL C inclusive, not the entire base.\n                breaking=h[i]>s[\'b\']\n                if breaking:\n                    depth=(s[\'b\']-s[\'c\'])/s[\'amp\']\n                    s[\'bc_bars\']=s[\'ci\']-s[\'bi\']\n                    if s[\'bc_bars\']<rules.bc_min_bars or depth<rules.retrace_min or (rules.require_time_divergence and s[\'bc_bars\']<=s[\'ab_bars\']):\n                        kill(\'B_break_before_BC_qualified\');state=None;continue\n                    s[\'phase\']=\'CD\';s[\'cd_start\']=i\n                    counts[\'bc_qualified\']+=1\n                elif i>s[\'bc_start\']:\n                    if l[i]<s[\'c\']:s[\'ci\']=i;s[\'c\']=l[i]\n                    s[\'base_bars\']+=1\n                    s[\'bc_bars\']=s[\'ci\']-s[\'bi\']\n                depth=(s[\'b\']-s[\'c\'])/s[\'amp\']\n                if (rules.boundary==\'wick\' and depth>rules.retrace_max) or c[i]<s[\'b\']-rules.retrace_max*s[\'amp\']:\n                    kill(\'invalid_BC_retrace\');state=None;continue\n                if s[\'base_bars\']>rules.max_bc_bars:\n                    kill(\'BC_expired\');state=None;continue\n                if s[\'phase\']==\'BC\':continue\n            bound=s[\'c\']+s[\'amp\']\n            # Strict AB>CD cannot touch equal-length boundary; close interpretation allows wick.\n            if (rules.boundary==\'wick\' and h[i]>=bound) or c[i]>=bound:\n                kill(\'invalid_AB_not_gt_CD\');state=None;continue\n            if s[\'phase\']==\'CD\' and l[i]<=s[\'c\']:\n                # C frozen at breakout; a breakout candle making a new C is rejected.\n                kill(\'invalid_CD_new_C\');state=None;continue\n            previous_high=s[\'highest\'];s[\'highest\']=max(s[\'highest\'],h[i])\n            if s[\'phase\']==\'ARMED\':\n                if i-s[\'signal_end\']>rules.max_armed_bars:\n                    kill(\'trigger_expired\');state=None;continue\n                av=ats[i]\n                if o[i]>=s[\'signal_low\'] and c[i]<s[\'signal_low\'] and o[i]-c[i]>=rules.trigger_body_atr*av:\n                    stop=s[\'highest\']+rules.stop_buffer_atr*av\n                    risk=stop-c[i]\n                    if risk>0:\n                        ev={\'time\':index[i],\'side\':-direction,\'distance\':risk,\'rr\':rules.rr,\n                            \'stop_price\':direction*stop,\'setup_id\':f\'{direction}:{index[s["bi"]].isoformat()}\',\n                            \'a_time\':index[s[\'ai\']],\'b_time\':index[s[\'bi\']],\'c_time\':index[s[\'ci\']],\n                            \'cd_start_time\':index[s[\'cd_start\']],\'signal_time\':index[s[\'signal_end\']],\n                            \'a\':direction*s[\'a\'],\'b\':direction*s[\'b\'],\'c\':direction*s[\'c\'],\n                            \'d\':direction*s[\'highest\'],\'signal_level\':direction*s[\'signal_low\'],\n                            \'ab_atr\':s[\'amp\']/s[\'atr\'],\'ab_efficiency\':s[\'ab_efficiency\'],\'ab_noise\':s[\'ab_noise\'],\n                            \'ab_bars\':s[\'ab_bars\'],\'bc_bars\':s[\'bc_bars\'],\'base_bars_after_C\':s[\'base_bars\']-s[\'bc_bars\'],\'retrace\':(s[\'b\']-s[\'c\'])/s[\'amp\'],\n                            \'cd_ab_ratio\':(s[\'highest\']-s[\'c\'])/s[\'amp\'],\n                            \'time_divergence\':float(s[\'bc_bars\']>s[\'ab_bars\']),\n                            \'signal_body_ratio\':s[\'body_ratio\'],\'signal_wick_ratio\':s[\'wick_ratio\'],\n                            \'signal_range_atr\':s[\'sig_range_atr\'],\'trigger_body_atr\':(o[i]-c[i])/av,\n                            \'risk_atr\':risk/av,\'fl_bars\':s[\'fl_bars\']}\n                        emitted.append(ev);counts[\'triggers\']+=1;consumed_b=s[\'bi\']\n                    state=None\n                continue\n            if s[\'fl_start\'] is None:\n                if h[i]>s[\'b\'] and h[i]>previous_high:\n                    s[\'fl_start\']=i\n                else:continue\n            k=s[\'fl_start\'];fl_n=i-k+1\n            if fl_n>(3 if rules.allow_fl else 1):\n                kill(\'invalid_signal_or_FL\');state=None;continue\n            high=float(np.max(h[k:i+1])); low=float(np.min(l[k:i+1]));rng=high-low\n            body=c[i]-o[k];upper=high-max(o[k],c[i]);pos=(s[\'b\']-low)/max(rng,1e-12)\n            body_ratio=body/max(rng,1e-12);wick_ratio=upper/max(rng,1e-12)\n            if body>0 and c[i]>s[\'b\'] and body_ratio>=rules.signal_body_ratio and wick_ratio<=rules.signal_wick_ratio and rng>=rules.signal_range_atr*ats[k] and rules.b_position_low<=pos<=rules.b_position_high and (fl_n==1 or high>h[k]):\n                s.update(phase=\'ARMED\',signal_end=i,signal_low=low,body_ratio=body_ratio,\n                         wick_ratio=wick_ratio,sig_range_atr=rng/ats[k],fl_bars=fl_n)\n                counts[\'signals_armed\']+=1\n                trace.append({\'time\':str(index[i]),\'direction\':direction,\'stage\':\'ARMED\',\'b_time\':str(index[s[\'bi\']])})\n    events=pd.DataFrame(emitted)\n    if len(events):\n        events=events.set_index(\'time\').sort_index(kind=\'stable\')\n        # A simultaneous opposite trigger is ambiguous; drop BOTH, not choose hindsight winner.\n        collision=events.index.duplicated(keep=False);counts[\'simultaneous_dropped\']=int(collision.sum())\n        events=events.loc[~collision]\n    else:events=pd.DataFrame(columns=EVENT_COLUMNS,index=pd.DatetimeIndex([],tz=\'UTC\',name=\'time\'))\n    return events, {\'rules\':asdict(rules),\'counts\':dict(counts),\'trace\':trace}\n', encoding='utf-8')
(PACKAGE / 'pivot_experiment.py').write_text('"""Experiments restricted to pivot settlement, never a search over unrelated strategies."""\nfrom dataclasses import replace, asdict\nfrom pathlib import Path\nimport hashlib, json, html, base64\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom .pivot import PivotRules, FEATURE_COLUMNS, detect_pivots, pivot_bars\nfrom .engine import Execution, backtest\nfrom .report import metrics\n\nPROTOCOL = {\'variants\':[\'wick_2.5R\',\'wick_3R\',\'close_2.5R\',\'close_3R\'],\n            \'baseline\':\'wick_2.5R\', \'chronology\':\'60% train / 20% validation / 20% test of elapsed UTC time\',\n            \'thresholds\':[.35,.40,.45,.50], \'winner_retention_min\':.95,\n            \'positive_pnl_retention_min\':.95, \'losers_removed_min\':.10,\n            \'minimum_train_labels\':100, \'minimum_validation_winners\':30,\n            \'minimum_validation_losers\':30, \'embargo_hours\':24,\n            \'seed\':8056, \'minimum_input_months_for_ml\':12,\n            \'target\':\'Mean net complete monthly return >=3%, equity DD <10%; >=24 OOS months for long-term review\',\n            \'sample_status\':\'ENGINEERING_ONLY\', \'no_passing_filter\':\'bypass, preserve baseline signals\',\n            \'no_variant_selection_on_test\':True}\n\n\ndef paired_retention(labels, keep):\n    labels=labels.copy(); keep=np.asarray(keep,dtype=bool)\n    win=labels.pnl>0; loss=labels.pnl<0\n    return {\'winners\':int(win.sum()),\'losers\':int(loss.sum()),\n        \'kept_winners\':int((win&keep).sum()),\'removed_winners\':int((win&~keep).sum()),\n        \'kept_losers\':int((loss&keep).sum()),\'removed_losers\':int((loss&~keep).sum()),\n        \'winner_retention\':float(keep[win].mean()) if win.any() else None,\n        \'positive_pnl_retention\':float(labels.loc[win&keep,\'pnl\'].sum()/labels.loc[win,\'pnl\'].sum()) if win.any() else None,\n        \'losers_removed_fraction\':float((~keep)[loss].mean()) if loss.any() else None,\n        \'removed_positive_pnl\':float(labels.loc[win&~keep,\'pnl\'].sum()),\n        \'avoided_negative_pnl\':float(-labels.loc[loss&~keep,\'pnl\'].sum()),\n        \'isolated_net_pnl_delta\':float(-labels.loc[~keep,\'pnl\'].sum()),\n        \'note\':\'Paired ISOLATED trades. Portfolio effect differs: removing a trade can allow a later entry.\'}\n\n\ndef isolated_labels(ticks,events,cfg):\n    """Labels use actual quote replay and absolute stop. Not vectorized candle approximations."""\n    records=[]\n    for j,(time,event) in enumerate(events.iterrows()):\n        end=time+pd.Timedelta(hours=cfg.max_hold_hours)+pd.Timedelta(minutes=5)\n        lo=ticks.index.searchsorted(time,side=\'left\'); hi=ticks.index.searchsorted(end,side=\'right\')\n        window=ticks.iloc[lo:hi]\n        if len(window)<2:continue\n        r=backtest(window,events.loc[[time]],cfg)\n        if len(r[\'trades\'])!=1:continue  # session/spread/risk/latency rejected => no training label\n        t=r[\'trades\'].iloc[0]\n        if int(t.reason)==6:continue # right-censored window/end-of-file is NOT a resolved training outcome\n        rec={f:float(event[f]) for f in FEATURE_COLUMNS}\n        rec.update(time=time,label_exit=t.exit_time,pnl=float(t.pnl),R=float(t.realized_R),\n                   side=int(event.side),reason=int(t.reason),setup_id=event.setup_id)\n        records.append(rec)\n    if records:return pd.DataFrame(records).set_index(\'time\').sort_index()\n    return pd.DataFrame(columns=[*FEATURE_COLUMNS,\'label_exit\',\'pnl\',\'R\',\'side\',\'reason\',\'setup_id\'],index=pd.DatetimeIndex([],tz=\'UTC\',name=\'time\'))\n\n\ndef nonoverlapping(labels):\n    # Independent exposure windows for fitting; purge is on label end, not rows.\n    indices=[];last=None\n    for i,(time,row) in enumerate(labels.iterrows()):\n        if last is None or time>last:\n            indices.append(i);last=row.label_exit\n    return labels.iloc[indices]\n\n\ndef fit_filter(labels,events,train_end,val_end,enough_history):\n    info={\'enabled\':False,\'reason\':\'INSUFFICIENT_HISTORY\',\'threshold\':None,\'training_labels\':0,\'validation_labels\':0}\n    if not enough_history:\n        info[\'available_development_labels\']=len(labels)\n        return None,None,pd.DataFrame(),info\n    train=labels.loc[(labels.index<train_end)&(labels.label_exit<train_end-pd.Timedelta(hours=PROTOCOL[\'embargo_hours\']))]\n    train=nonoverlapping(train).dropna(subset=FEATURE_COLUMNS)\n    val=labels.loc[(labels.index>=train_end)&(labels.index<val_end)&(labels.label_exit<val_end-pd.Timedelta(hours=PROTOCOL[\'embargo_hours\']))].dropna(subset=FEATURE_COLUMNS)\n    info.update(training_labels=len(train),validation_labels=len(val))\n    if len(train)<PROTOCOL[\'minimum_train_labels\'] or (train.pnl>0).nunique()<2:\n        info[\'reason\']=\'INSUFFICIENT_PURGED_TRAINING_LABELS\';return None,None,pd.DataFrame(),info\n    if int((val.pnl>0).sum())<PROTOCOL[\'minimum_validation_winners\'] or int((val.pnl<0).sum())<PROTOCOL[\'minimum_validation_losers\']:\n        info[\'reason\']=\'INSUFFICIENT_VALIDATION_WINNERS_AND_LOSERS\';return None,None,pd.DataFrame(),info\n    model=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=500,random_state=8056))\n    model.fit(train[FEATURE_COLUMNS].astype(float),(train.pnl>0).astype(int))\n    probs=model.predict_proba(val[FEATURE_COLUMNS].astype(float))[:,1]\n    rows=[];best=None;best_gain=0.\n    for threshold in PROTOCOL[\'thresholds\']:\n        m=paired_retention(val,probs>=threshold)\n        passes=m[\'winner_retention\']>=PROTOCOL[\'winner_retention_min\'] and m[\'positive_pnl_retention\']>=PROTOCOL[\'positive_pnl_retention_min\'] and m[\'losers_removed_fraction\']>=PROTOCOL[\'losers_removed_min\'] and m[\'isolated_net_pnl_delta\']>0\n        rows.append({\'threshold\':threshold,\'eligible\':passes,**m})\n        if passes and m[\'isolated_net_pnl_delta\']>best_gain:best=threshold;best_gain=m[\'isolated_net_pnl_delta\']\n    if best is None:\n        info[\'reason\']=\'NO_THRESHOLD_PRESERVES_VALIDATION_WINNERS\';return None,None,pd.DataFrame(rows),info\n    # NO refit on validation: keep scaler/coefficients used to choose threshold.\n    info.update(enabled=True,reason=\'VALIDATION_ACCEPTED_NOT_GUARANTEED\',threshold=best,\n                coefficients=model[-1].coef_.tolist(),intercept=model[-1].intercept_.tolist(),\n                scaler_mean=model[0].mean_.tolist(),scaler_scale=model[0].scale_.tolist())\n    return model,best,pd.DataFrame(rows),info\n\n\ndef plot_setup(bars,event,time,path):\n    start=event.a_time-pd.Timedelta(minutes=10);end=time+pd.Timedelta(minutes=20)\n    b=bars.loc[start:end]; fig,ax=plt.subplots(figsize=(12,4));axis=np.arange(len(b))\n    up=b.close>=b.open\n    for j,(_,row) in enumerate(b.iterrows()):\n        color=\'#138f70\' if up.iloc[j] else \'#c23c4b\'\n        ax.vlines(j,row.low,row.high,color=color)\n        ax.bar(j,abs(row.close-row.open),bottom=min(row.close,row.open),width=.65,color=color)\n    positions=[b.index.searchsorted(t) for t in [event.a_time,event.b_time,event.c_time,event.signal_time]]\n    prices=[event.a,event.b,event.c,event.d]\n    ax.plot(positions,prices,\'o-\',color=\'#1764b4\')\n    for j,p,label in zip(positions,prices,[\'A\',\'B\',\'C\',\'D (max to trigger)\']):ax.annotate(label,(j,p))\n    ax.axhline(event.signal_level,color=\'orange\',ls=\'--\',label=\'signal low/high\')\n    ax.axhline(event.stop_price,color=\'red\',ls=\':\',label=\'absolute stop\')\n    ax.axvline(b.index.searchsorted(time),color=\'black\',ls=\'--\',label=\'trigger CLOSE (entry later tick)\')\n    ax.set_title(f\'Automatic GBPUSD M1 pivot — {time}; NOT a DAX screenshot reproduction\')\n    ax.set_xticks(axis[::max(1,len(b)//10)]);ax.set_xticklabels([str(t)[11:16] for t in b.index[::max(1,len(b)//10)]])\n    ax.legend();fig.tight_layout();fig.savefig(path);plt.close(fig)\n\n\ndef run_pivot_experiment(ticks,audit,cfg=Execution(),folder=\'artifacts/pivot\',mode=\'sample\',bars=None):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    bars=pivot_bars(ticks) if bars is None else bars\n    start,end=ticks.index[0],ticks.index[-1]\n    train_end=start+(end-start)*.60;test_start=start+(end-start)*.80\n    test_ticks=ticks.loc[ticks.index>=test_start]\n    if len(test_ticks)<2:raise ValueError(\'Test has insufficient quotes\')\n    input_days=(end-start).total_seconds()/86400\n    histories={};all_events={};counts=[]\n    for boundary in [\'wick\',\'close\']:\n        e,log=detect_pivots(bars,replace(PivotRules(),boundary=boundary));all_events[boundary]=e\n        histories[boundary]=log\n        (folder/f\'trace_{boundary}.json\').write_text(json.dumps(log,ensure_ascii=False,indent=2))\n        e.to_csv(folder/f\'all_events_{boundary}.csv\')\n        counts.append({\'boundary\':boundary,**log[\'counts\']})\n    # Fixed baseline; four variants are sensitivity diagnostics, NOT best-of-four selection.\n    base_events=all_events[\'wick\'].copy()\n    # Drop pending triggers whose setup started before the test boundary from training LABELS\n    # only labels are purged; causal setup context may span split just as live trading does.\n    dev_events=base_events.loc[base_events.index<test_start]\n    labels=isolated_labels(ticks.loc[ticks.index<test_start],dev_events,cfg)\n    model,threshold,validation,filter_info=fit_filter(labels,dev_events,train_end,test_start,\n                                                    enough_history=(mode!=\'sample\' and input_days>=365))\n    frozen={\'protocol\':PROTOCOL,\'baseline_rules\':asdict(PivotRules()),\'execution\':asdict(cfg),\n            \'test_start\':str(test_start),\'train_end\':str(train_end),\'ml\':filter_info,\n            \'data_sha256\':audit.get(\'sha256\',audit.get(\'manifest\'))}\n    frozen[\'sha256\']=hashlib.sha256(json.dumps(frozen,sort_keys=True,default=str).encode()).hexdigest()\n    (folder/\'frozen_before_test.json\').write_text(json.dumps(frozen,indent=2,default=str))\n    rows=[];results={};smoke=[]\n    for boundary in [\'wick\',\'close\']:\n        for rr in [2.5,3.]:\n            name=f\'{boundary}_{rr:g}R\';events=all_events[boundary].copy();events[\'rr\']=rr\n            s=events.loc[events.index>=test_start]\n            result=backtest(test_ticks,s,cfg);stats,monthly,daily,bootstrap=metrics(result)\n            rows.append({\'variant\':name,\'role\':\'OOS sensitivity; NOT selected\',**stats});results[name]=result\n            result[\'trades\'].to_csv(folder/f\'test_trades_{name}.csv\',index=False)\n            monthly.to_csv(folder/f\'test_months_{name}.csv\',index=False)\n            result[\'equity\'].to_frame().to_parquet(folder/f\'test_tick_equity_{name}.parquet\')\n            if mode==\'sample\':\n                r=backtest(ticks,events,cfg)\n                r[\'trades\'].to_csv(folder/f\'smoke_trades_{name}.csv\',index=False)\n                smoke.append({\'variant\':name,\'role\':\'ALL WEEK SMOKE — NOT OOS\',**metrics(r)[0]})\n    test_events=base_events.loc[base_events.index>=test_start]\n    keep=np.ones(len(test_events),dtype=bool)\n    if model is not None and len(test_events):\n        keep=model.predict_proba(test_events[FEATURE_COLUMNS].astype(float))[:,1]>=threshold\n    filtered=backtest(test_ticks,test_events.loc[keep],cfg);results[\'wick_2.5R_ML\']=filtered\n    rows.append({\'variant\':\'wick_2.5R_ML\',\'role\':\'OOS frozen filter or bypass\',**metrics(filtered)[0]})\n    filtered[\'trades\'].to_csv(folder/\'test_trades_ML.csv\',index=False)\n    # Isolated OOS labels ONLY now, after frozen model/threshold. Never used to repair filtering.\n    test_events.assign(ML_keep=keep).to_csv(folder/\'test_signal_decisions.csv\')\n    test_labels=isolated_labels(test_ticks,test_events,cfg)\n    label_keep=np.ones(len(test_labels),dtype=bool)\n    if model is not None and len(test_labels):label_keep=model.predict_proba(test_labels[FEATURE_COLUMNS].astype(float))[:,1]>=threshold\n    preservation=paired_retention(test_labels,label_keep)\n    if len(test_labels):\n        test_labels[\'kept\']=label_keep\n        test_labels.to_csv(folder/\'paired_OOS_winners_losers.csv\')\n    stress={}\n    for name,stress_cfg in [(\'spread1.5_slip2\',replace(cfg,spread_multiplier=1.5,slippage=cfg.slippage*2)),\n                            (\'spread2_slip3_latency1s\',replace(cfg,spread_multiplier=2.,slippage=cfg.slippage*3,latency_ms=1000))]:\n        stress[name]=metrics(backtest(test_ticks,test_events.loc[keep],stress_cfg))[0]\n    filter_months=metrics(filtered)[1]; filter_bootstrap=metrics(filtered)[3]\n    metrics_df=pd.DataFrame(rows);metrics_df.to_csv(folder/\'test_comparison.csv\',index=False)\n    validation.to_csv(folder/\'ML_validation_thresholds.csv\',index=False)\n    pd.DataFrame(counts).to_csv(folder/\'funnel.csv\',index=False)\n    pd.DataFrame(smoke).to_csv(folder/\'sample_full_week.csv\',index=False)\n    # Predeclared rule ablations on DEVELOPMENT only, never override baseline on test.\n    ablations=[]\n    for name,rule in [\n        (\'body_ratio_0.50\',replace(PivotRules(),signal_body_ratio=.50)),\n        (\'body_ratio_0.70\',replace(PivotRules(),signal_body_ratio=.70)),\n        (\'no_FL\',replace(PivotRules(),allow_fl=False)),\n        (\'require_BC_time_divergence\',replace(PivotRules(),require_time_divergence=True)),\n    ]:\n        dev_bars=bars.loc[bars.index<test_start]\n        dev_s,dev_log=detect_pivots(dev_bars,rule)\n        dev_r=backtest(ticks.loc[ticks.index<test_start],dev_s,cfg)\n        ablations.append({\'ablation\':name,\'role\':\'DEVELOPMENT ONLY — NOT SELECTED\',\n                          \'triggers\':len(dev_s),**metrics(dev_r)[0]})\n    pd.DataFrame(ablations).to_csv(folder/\'development_rule_ablations.csv\',index=False)\n    smoke_stress={}\n    if mode==\'sample\':\n        for name,stress_cfg in [(\'spread1.5_slip2\',replace(cfg,spread_multiplier=1.5,slippage=cfg.slippage*2)),\n                                (\'spread2_slip3_latency1s\',replace(cfg,spread_multiplier=2.,slippage=cfg.slippage*3,latency_ms=1000))]:\n            smoke_stress[name]=metrics(backtest(ticks,base_events,stress_cfg))[0]\n    # A time-divergence-only hypothesis reported on development only; not selected on OOS.\n    td=dev_events.loc[dev_events.time_divergence.astype(float)==1] if len(dev_events) else dev_events\n    dev_ticks=ticks.loc[ticks.index<test_start]\n    td_stats=metrics(backtest(dev_ticks,td,cfg))[0]\n    report={\'status\':\'INSUFFICIENT_EVIDENCE\' if mode==\'sample\' else \'OOS_RESEARCH_NOT_GUARANTEED\',\n        \'pdf_rules_operational_not_confirmed\':True,\'exact_DAX_image_signal_match_verified\':False,\n        \'long_term_monthly_target_verified\':False,\'mean_monthly_target_observed\':False,\n        \'full_history_validation_complete\':False,\'mode\':mode,\'data_audit\':audit,\n        \'development_rule_ablation_trials\':len(ablations),\'development_ablations\':ablations,\n        \'sample_full_week_cost_stress_NOT_OOS\':smoke_stress,\'rule_variants_trial_count\':4,\'ML_threshold_trials\':len(PROTOCOL[\'thresholds\']),\n        \'closed_bid_bar_count\':len(bars),\'input_days\':input_days,\'splits\':{\'train_end\':str(train_end),\'test_start\':str(test_start)},\n        \'frozen_sha256\':frozen[\'sha256\'],\'ML\':filter_info,\'paired_test_preservation\':preservation,\n        \'complete_filtered_test_months\':int(filter_months.complete.sum()),\n        \'filtered_test_bootstrap\':filter_bootstrap, \'stress_filtered_baseline\':stress,\'optional_time_divergence_development_only\':td_stats,\n        \'oos_results\':metrics_df.where(pd.notna(metrics_df),None).to_dict(\'records\'),\n        \'limitations\':[\'No matching DAX/FXCM raw candles/ticks or chart timezone supplied. Image audit is qualitative only.\',\n                      \'Only GBPUSD real bid/ask mirror data executed here; not DAX profitability or image reproduction.\',\n                      \'Numerical candle thresholds, impulse segmentation, B-position band and expiry are explicit operational assumptions.\',\n                      \'C is provisionally updated then frozen at B breakout; BC duration is B exclusive through final C inclusive. Base after C logged separately.\',\n                      \'Fixed TP 2.5/3R per user; screenshot name returns are NOT training labels or expected exits.\',\n                      \'Validation preservation is not guaranteed out of sample. Removed winners and positive PnL must be reported.\',\n                      \'Same-tick liquidity, market impact, swaps and cross-currency account conversion not modeled; risk breakers can gap.\',\n                      \'OHLC is for signals ONLY. Prices and absolute stops executed with real bid/ask on strictly later quotes.\',\n                      \'Short sample cannot fit ML or establish monthly target. Bypass does not prove an ML improvement.\',\n                      \'Four OOS variants are sensitivity tests, not independent untouched proofs after human comparison.\']}\n    # Convert NaN to null for honest JSON.\n    text=json.dumps(report,indent=2,ensure_ascii=False,default=str)\n    text=text.replace(\'NaN\',\'null\');(folder/\'summary.json\').write_text(text)\n    fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)\n    for name,r in results.items():\n        eq=r[\'equity\'].groupby(level=0).last().resample(\'1min\').last().dropna()\n        axes[0].plot(eq.index,eq,label=name)\n        axes[1].plot(eq.index,100*(1-eq/eq.cummax().clip(lower=cfg.initial_cash)),label=name)\n    axes[0].set_ylabel(\'Liquidation equity\');axes[1].set_ylabel(\'DD %\');axes[0].legend()\n    fig.tight_layout();fig.savefig(folder/\'comparison.png\');plt.close(fig)\n    illustrations=[]\n    for k,(time,event) in enumerate(base_events.head(4).iterrows()):\n        p=folder/f\'auto_setup_{k+1}.png\';plot_setup(bars,event,time,p);illustrations.append(p)\n    image=lambda p:\'<img style="max-width:100%" src="data:image/png;base64,\'+base64.b64encode(p.read_bytes()).decode()+\'">\'\n    body=\'<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><title>Pivot settlement audit</title><style>body{font-family:system-ui;max-width:1200px;margin:24px auto;padding:20px}table{direction:ltr;font-size:12px;border-collapse:collapse}td,th{border:1px solid #bbb;padding:6px}pre{direction:ltr;text-align:left;white-space:pre-wrap}</style>\'\n    body+=\'<h1>پیوت تسویه — گزارش پژوهش</h1><p>تطبیق عددی تصاویر DAX انجام نشده؛ این نمودارهای خودکار GBPUSD هستند. نتیجه کوتاه، تضمین سود ماهانه نیست.</p>\'\n    body+=image(folder/\'comparison.png\')+\'<h2>حساسیت خارج از نمونه — بدون انتخاب برنده</h2>\'+metrics_df.to_html(index=False)\n    body+=\'<h2>آزمون مهندسی کل هفته — نه خارج از نمونه</h2>\'+pd.DataFrame(smoke).to_html(index=False)\n    body+=\'<h2>تغییرات همان منطق — فقط بخش توسعه، بدون انتخاب روی تست</h2>\'+pd.DataFrame(ablations).to_html(index=False)\n    body+=\'<h2>حفظ سودها و حذف خطاها</h2><pre>\'+html.escape(json.dumps(preservation,ensure_ascii=False,indent=2))+\'</pre>\'\n    body+=\'<h2>نمونه\u200cهای تشخیص خودکار با A/B/C/D</h2>\'+\'\'.join(image(p) for p in illustrations)\n    body+=\'<h2>فرض\u200cها و گزارش جامع</h2><pre>\'+html.escape(text)+\'</pre></html>\'\n    (folder/\'report.html\').write_text(body,encoding=\'utf-8\')\n    return report,metrics_df,results\n', encoding='utf-8')
(PACKAGE / 'pivot_match.py').write_text('"""Compare exact OHLC exports to user annotations; proximity is not proof of fidelity."""\nfrom pathlib import Path\nimport pandas as pd\nfrom .pivot import validate_bars,detect_pivots,PivotRules\n\n\ndef read_chart_bars(path, source_timezone, timestamp_is_open=True):\n    f=pd.read_csv(path);f.columns=[str(c).lower() for c in f.columns]\n    if \'time\' in f and \'timestamp\' not in f:f=f.rename(columns={\'time\':\'timestamp\'})\n    if \'timestamp\' not in f:raise ValueError(\'CSV requires timestamp/time and open/high/low/close\')\n    raw=f.pop(\'timestamp\')\n    if pd.api.types.is_numeric_dtype(raw):idx=pd.DatetimeIndex(pd.to_datetime(raw,unit=\'s\',utc=True))\n    else:idx=pd.DatetimeIndex(pd.to_datetime(raw,format=\'mixed\'))\n    if idx.tz is None:idx=idx.tz_localize(source_timezone,ambiguous=\'raise\',nonexistent=\'raise\')\n    idx=idx.tz_convert(\'UTC\')\n    if timestamp_is_open:idx=idx+pd.Timedelta(minutes=1)\n    f.index=idx;return validate_bars(f)\n\n\ndef compare_images(bars,annotations,chart_timezone,rules=PivotRules(),tolerance_minutes=2):\n    events,log=detect_pivots(bars,rules)\n    records=[]\n    for _,a in annotations.iterrows():\n        time=a.get(\'approx_trigger_chart_local\')\n        base={\'image\':a.image,\'boundary\':rules.boundary,\'expected_side\':a.side,\'status\':\'UNVERIFIABLE\',\n              \'nearest_trigger\':None,\'delta_minutes\':None,\'note\':\'Approximate screenshot timing; needs manual OHLC review.\'}\n        if pd.isna(time) or not time:\n            base[\'note\']=\'No readable trigger timestamp in screenshot\';records.append(base);continue\n        expected=pd.Timestamp(str(a.chart_date)+\' \'+str(time)).tz_localize(chart_timezone).tz_convert(\'UTC\')\n        # Chart mark is approximate entry/open time, detector event is trigger candle close.\n        subset=events.loc[(events.index>=expected-pd.Timedelta(minutes=tolerance_minutes)) &\n                          (events.index<=expected+pd.Timedelta(minutes=tolerance_minutes))]\n        if not (bars.index.min()<=expected<=bars.index.max()):\n            base[\'status\']=\'DATA_DATE_NOT_COVERED\'\n        elif len(subset)==0:base[\'status\']=\'NO_NEARBY_TRIGGER\'\n        else:\n            side=-1 if a.side==\'sell\' else 1;same=subset.loc[subset.side==side]\n            if len(same):\n                delta=abs((same.index-expected).total_seconds()/60);j=int(delta.argmin())\n                base.update(status=\'NEARBY_SAME_DIRECTION_NOT_EXACT_PROOF\',nearest_trigger=str(same.index[j]),delta_minutes=float(delta[j]))\n            else:base[\'status\']=\'ONLY_OPPOSITE_TRIGGER_NEARBY\'\n        records.append(base)\n    return pd.DataFrame(records),events,log\n', encoding='utf-8')
(PACKAGE / 'pivot_long.py').write_text('"""Long-history quote validation and year/quarter financial audit on supplied real ticks."""\nfrom pathlib import Path\nimport json\nimport pandas as pd\nfrom .pivot_experiment import run_pivot_experiment\nfrom .engine import Execution\n\n\ndef long_quote_gate(ticks, minimum_days=365):\n    if not {\'bid\',\'ask\'}<=set(ticks):raise ValueError(\'Real bid AND ask required; OHLC cannot satisfy long tick backtest\')\n    if not isinstance(ticks.index,pd.DatetimeIndex) or ticks.index.tz is None or not ticks.index.is_monotonic_increasing:\n        raise ValueError(\'Sorted timezone-aware real quotes required\')\n    span=(ticks.index[-1]-ticks.index[0]).total_seconds()/86400\n    # Allow weekend start/end in a nominal calendar year; cannot certify broker provenance.\n    if span<minimum_days-7:raise ValueError(f\'Need approximately {minimum_days} days real quotes; have {span:.2f}\')\n    dates=pd.date_range(ticks.index[0].normalize(),ticks.index[-1].normalize(),freq=\'B\',tz=\'UTC\')\n    missing=dates.difference(ticks.index.normalize().unique())\n    fraction=len(missing)/max(1,len(dates))\n    if fraction>.03:raise ValueError(f\'Missing {len(missing)} weekday dates ({fraction:.1%}); repair coverage first\')\n    return {\'span_days\':span,\'missing_weekdays\':[str(d.date()) for d in missing],\n            \'missing_weekday_fraction\':fraction,\'coverage_gate_passed\':True,\n            \'warning\':\'Gate checks coverage only, not quote authenticity or suitability for execution.\'}\n\n\ndef run_long_pivot(ticks,audit,cfg=Execution(),folder=\'artifacts/pivot_long\',minimum_days=365):\n    coverage=long_quote_gate(ticks,minimum_days)\n    p=Path(folder);p.mkdir(parents=True,exist_ok=True)\n    audit={**audit,\'long_coverage\':coverage}\n    # Frozen 60/20/20 initial experiment; no repeated changes to its final 20%.\n    report,comparison,results=run_pivot_experiment(ticks,audit,cfg,folder=folder,mode=\'research\')\n    # Calendar-year and quarter net equity changes (NOT independent reinitialised accounts).\n    rows=[]\n    for name,result in results.items():\n        eq=result[\'equity\'].groupby(level=0).last()\n        for freq in [\'YE\',\'QE\']:\n            marks=eq.resample(freq).last();previous=cfg.initial_cash\n            for time,value in marks.items():\n                rows.append({\'variant\':name,\'period_end\':str(time),\'frequency\':freq,\n                             \'net_equity_return\':float(value/previous-1),\n                             \'partial_boundary_period_possible\':True,\n                             \'note\':\'Sequential OOS account, not a fresh independent year/quarter backtest.\'})\n                previous=value\n    pd.DataFrame(rows).to_csv(p/\'OOS_year_quarter_equity.csv\',index=False)\n    report[\'long_quote_coverage\']=coverage\n    report[\'stability_verified\']=False\n    report[\'required_next_stage\']=\'Multiple chronological walk-forward folds and a fresh untouched future holdout; no stability claim from one split.\'\n    (p/\'long_audit.json\').write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str).replace(\'NaN\',\'null\'))\n    return report,comparison,results\n', encoding='utf-8')

from hippo_lab.engine import Execution,backtest
from hippo_lab.pivot import PivotRules,pivot_bars,detect_pivots,FEATURE_COLUMNS
from hippo_lab.pivot_experiment import run_pivot_experiment
from hippo_lab.pivot_match import read_chart_bars,compare_images
from hippo_lab.pivot_long import run_long_pivot
execution=Execution(pair=PAIR,account_currency=ACCOUNT_CURRENCY)
bars=pivot_bars(ticks,'1min')
print('Closed BID M1 bars:',len(bars))
print(execution)
print(PivotRules())
# DAX/CFD contract sizing is NOT implemented in this FX account engine.
# Costs/account units must be set before outcomes. No leverage tuning to reach target.


## آزمون علیت و حسابداری

مصنوعی زیر فقط fixture نرم‌افزار است و هیچ بازده آن نتیجه مالی نیست. روی همین داده باید AB/BC/شکست B/تریگر به ترتیب تشخیص داده شود، سیگنال فروش فقط بعد از شکست کف سیگنال صادر شود و افزودن آینده نتیجه گذشته را عوض نکند.


In [ ]:
# @title ۳ — آزمون بازتولیدپذیری منطق
from dataclasses import replace
rows=[(100,100.5,99.5,100)]*8
rows += [(100,102.2,99.8,102),(102,104.2,101.8,104),(104,106.2,103.8,106),
         (106,106.1,105,105.5),(105.5,105.8,104.8,105),(105,105.2,104,104.5),
         (104.5,107,104.3,106.8),(106.8,107,103.5,103.8),(103.8,104,103,103.2)]
ix=pd.date_range('2024-01-02 14:00',periods=len(rows),freq='1min',tz='UTC')
fixture=pd.DataFrame(rows,columns=['open','high','low','close'],index=ix)
check_rule=replace(PivotRules(),atr_period=2,ab_max_bars=3,ab_min_median_range_atr=.1,
                   signal_range_atr=.1,trigger_body_atr=.1)
events,trace=detect_pivots(fixture,check_rule)
assert len(events)==1 and events.iloc[0].side==-1 and events.index[0]==ix[15]
assert events.iloc[0].signal_time==ix[14] and events.iloc[0].bc_bars==3
past,_=detect_pivots(fixture.iloc[:16],check_rule)
pd.testing.assert_frame_equal(events,past)
print('PASS: PDF phase ordering, strict reversal timing, BC duration, prefix invariance')


## تطبیق تصاویر: مرز شواهد

جدول زیر برداشت دستی همه نمونه‌هاست. `QUALITATIVE_ONLY_NOT_NUMERIC_MATCH` یعنی **کد روی داده همین تصاویر اجرا نشده**. برای مقایسه واقعی، CSV خروجی OHLC همان FXCM_GER30 با زمان و منطقه زمانی لازم است. ابزار تطبیق اختیاری در همین نوت‌بوک وجود دارد؛ نزدیکی زمان/جهت تنها تست اولیه است و اثبات یکسان‌بودن A/B/C/D نیست.


In [ ]:
# @title ۴ — مرور تصاویر و تطبیق اختیاری داده همان نمودار
image_annotations=pd.DataFrame(json.loads('[{"image":"DAX-2026-01-08-001-8.9R.png","chart_date":"2026-01-08","approx_trigger_chart_local":"07:57","side":"sell","structure_observed":"AB up; long base; breakout and reversal; highlighted signal range","caveat":"B around 07:22; trigger 07:57","filename_claimed_R":"8.9","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"ac5fd4af1b6408aedb8c56e9408e66a53d5e993faa906956c7f14f56dafdfe98","raw_quote_match_available":false},{"image":"DAX-2026-03-31-002-3.2R.png","chart_date":"2026-03-31","approx_trigger_chart_local":"13:39","side":"sell","structure_observed":"AB up; brief BC then B breakout and bearish body trigger","caveat":"B around 13:30","filename_claimed_R":"3.2","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"c1256c205f13f650ab4848f520c8eee2cdd9aeab699eae30dc0277f84b137f0b","raw_quote_match_available":false},{"image":"DAX-2026-04-26-001-5.3R.png","chart_date":"2026-04-23","approx_trigger_chart_local":"15:06","side":"buy","structure_observed":"AB down; B lower break; bullish reversal","caveat":"Filename date 04-26 differs from Thu 23 Apr chart","filename_claimed_R":"5.3","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"54cac8c2473d102b0f0e62597a438fdaf871f751fa4172f4e39f81c40a70f302","raw_quote_match_available":false},{"image":"DAX-2026-05-05-001-14.6R.png","chart_date":"2026-05-06","approx_trigger_chart_local":"09:03","side":"buy","structure_observed":"AB down; base; second extension down then reversal","caveat":"Filename date 05-05 differs from Wed 06 May chart","filename_claimed_R":"14.6","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"a18244d1e727c9a90a03ae3b5a84e8f930f154d7b06b52afce140ab95aa31df3","raw_quote_match_available":false},{"image":"DAX-2026-05-08-001-1.3R.png","chart_date":"2026-05-08","approx_trigger_chart_local":null,"side":"sell","structure_observed":"AB up; B break around 17:00; bearish trigger before 17:18","caveat":"17:18 label is later endpoint; exact entry minute unreadable","filename_claimed_R":"1.3","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"7b891a643503cc5126deeba0462321d3d797d012bac54dcfd755d4334fb58c8a","raw_quote_match_available":false},{"image":"DAX-2026-05-11-001-1.6R.png","chart_date":"2026-05-11","approx_trigger_chart_local":"06:02","side":"sell","structure_observed":"AB up; base; B extension; large bearish trigger","caveat":"Later pullback near stop; exact fill unknown","filename_claimed_R":"1.6","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"c927823d22b7ce74d60bd621346bdd4ccd9f373a7742f8bccce260551a807db4","raw_quote_match_available":false},{"image":"DAX-2026-05-13-001-13.1R.png","chart_date":"2026-05-13","approx_trigger_chart_local":"06:26","side":"buy","structure_observed":"AB down; lower B break; bullish trigger","caveat":"Large retrace after entry; cannot infer stop survival from filename","filename_claimed_R":"13.1","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"b73356023ece0a90494986b99ac91d1800b3ed58da80dc28ae5034b01ccff5cd","raw_quote_match_available":false},{"image":"DAX-2026-05-13-002-2.2R.png","chart_date":"2026-05-13","approx_trigger_chart_local":"11:04","side":"buy","structure_observed":"AB down; extended base; lower break; bullish trigger","caveat":"TP\\/exit not determined by chart path alone","filename_claimed_R":"2.2","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"422a1150b3aab7cc7d024cd7b51f774aec258c4c7287efea0c6bce55eaa78f7c","raw_quote_match_available":false},{"image":"DAX-2026-05-15-001-5.5R.png","chart_date":"2026-05-15","approx_trigger_chart_local":"07:32","side":"sell","structure_observed":"AB up; long base; higher extension; bearish reversal","caveat":"Potential multi-bar signal; raw OHLC needed","filename_claimed_R":"5.5","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"fb1c007eed40b3a1c4c7f6be27f9a9c1c456144ddce3ac89a9ccc656580a77a1","raw_quote_match_available":false},{"image":"DAX-2026-05-19-001-2.1R.png","chart_date":"2026-05-19","approx_trigger_chart_local":"11:32","side":"sell","structure_observed":"AB up in prior rising regime; extension then bearish trigger","caveat":"System is reversal, not continuation","filename_claimed_R":"2.1","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"4b5096c2212d8f3905c2c0d288cd92778a3d37b216d334c38b2613c92cbe8625","raw_quote_match_available":false},{"image":"DAX-2026-05-20-001-1.5R.png","chart_date":"2026-05-20","approx_trigger_chart_local":"11:06","side":"sell","structure_observed":"AB up; BC deep; highlighted liquidity box; two entry rectangles","caveat":"Multiple discretionary entries\\/re-entry not defined by PDF","filename_claimed_R":"1.5","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"d8de707b5b43931f2db81a366ea3153fda1115ecdbc6099b0ff6eef7a14d1e05","raw_quote_match_available":false},{"image":"DAX-2026-05-21-001-SL.png","chart_date":"2026-05-21","approx_trigger_chart_local":"09:43","side":"sell","structure_observed":"AB up; base; higher extension then bearish trigger","caveat":"Stop hit by rebound even though later price falls; outcome path matters","filename_claimed_R":"SL","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"703815daf4d57a6957855256e030feb83a7d4688cdea1830ce33f3a524a9bbde","raw_quote_match_available":false},{"image":"DAX-2026-05-22-001-1R.png","chart_date":"2026-05-22","approx_trigger_chart_local":null,"side":"sell","structure_observed":"AB up; highlighted signal box; downside break around 05:30","caveat":"Exact trigger minute not labelled","filename_claimed_R":"1","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"b00289ce1163adb3ab0600b99e4609df7e39485e73d81d215558a1c0f718533c","raw_quote_match_available":false},{"image":"DAX-2026-05-26-001-3.8R.png","chart_date":"2026-05-26","approx_trigger_chart_local":"14:45","side":"sell","structure_observed":"AB up; brief base; higher sweep then bearish trigger","caveat":"Later reversal could stop tighter exits","filename_claimed_R":"3.8","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"cdc712a90c40398377cceea65a83a2c3bd838bc25e63029496c13a181f375316","raw_quote_match_available":false},{"image":"DAX-2026-05-28-001-12.9R.png","chart_date":"2026-05-28","approx_trigger_chart_local":"07:53","side":"buy","structure_observed":"AB down; base; lower extension then bullish trigger","caveat":"Strong later expansion does not specify exit rule","filename_claimed_R":"12.9","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"8cb7e4584084a163f1e0264cddff31f08d2777036c0b2adefda6b99e3de48295","raw_quote_match_available":false},{"image":"DAX-2026-06-01-001-4.7R.png","chart_date":"2026-06-01","approx_trigger_chart_local":"13:36","side":"buy","structure_observed":"AB down; lower B break; bullish trigger","caveat":"Late session reversal after chosen endpoint","filename_claimed_R":"4.7","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"90ec6d34cd89bee98a7b2f68a1a81abcff702901182071c193c7be97328199ca","raw_quote_match_available":false},{"image":"DAX-2026-06-02-001-10R.png","chart_date":"2026-06-02","approx_trigger_chart_local":"13:55","side":"sell","structure_observed":"AB up; base; B higher break then bearish trigger","caveat":"Large later decline; screenshot return not fixed-TP result","filename_claimed_R":"10","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"43d0d722da82259329076141c69954360f35af06dcb84718d217ef2049476d6f","raw_quote_match_available":false},{"image":"DAX-2026-06-02-002-1R.png","chart_date":"2026-06-02","approx_trigger_chart_local":"17:06","side":"sell","structure_observed":"AB up; B extension then bearish trigger","caveat":"Long hold\\/rebound; no trailing rule defined","filename_claimed_R":"1","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"a7baf7eeafda5ef682a06ec8395e5fd8f215c7668b3e629d0b8cc988c3d7c801","raw_quote_match_available":false},{"image":"DAX-2026-06-03-001-1.6R.png","chart_date":"2026-06-03","approx_trigger_chart_local":"16:09","side":"buy","structure_observed":"AB down; base; lower extension then bullish trigger","caveat":"Endpoint 16:37; costs and exact R not reproducible","filename_claimed_R":"1.6","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"785cd627a3260feca9fadbde42782135a3d0fef0643db246d03d8d11a950a281","raw_quote_match_available":false},{"image":"DAX-2026-06-05-001-1.6R.png","chart_date":"2026-06-05","approx_trigger_chart_local":"09:49","side":"sell","structure_observed":"AB up; base; extension then bearish trigger","caveat":"Duplicated byte-identically by filename (1)","filename_claimed_R":"1.6","audit_status":"QUALITATIVE_ONLY_NOT_NUMERIC_MATCH","sha256":"ef11e3e54ba55f08c21f081fe49a40c8cd8ee73bced3262dd8e1ea69344989e8","raw_quote_match_available":false}]'))

display(image_annotations[['image','chart_date','approx_trigger_chart_local','side','structure_observed','caveat','audit_status']])
CHART_CSV = '' # @param {type:'string'}
CHART_TIMEZONE = 'Europe/Berlin' # @param {type:'string'}
CHART_TIMESTAMP_IS_OPEN = True # @param {type:'boolean'}
# Default timezone above is ONLY a placeholder input, NOT known screenshot timezone.
# Fill CHART_CSV only with actual matching symbol data and correct chart timezone.
if CHART_CSV:
    chart_bars=read_chart_bars(CHART_CSV,CHART_TIMEZONE,CHART_TIMESTAMP_IS_OPEN)
    comparisons=[]
    for boundary in ['wick','close']:
        match,chart_events,chart_trace=compare_images(chart_bars,image_annotations,CHART_TIMEZONE,
                                                     replace(PivotRules(),boundary=boundary))
        comparisons.append(match)
    chart_comparison=pd.concat(comparisons,ignore_index=True)
    display(chart_comparison)
    Path('artifacts/pivot').mkdir(parents=True,exist_ok=True)
    chart_comparison.to_csv('artifacts/pivot/image_OHLC_match.csv',index=False)
else:
    print('Exact DAX screenshot reproduction: NOT VERIFIED — no matching OHLC supplied')


## ارزیابی تاریخچه بلند و پروتکل آزمایش

در حالت local/histdata اکنون حداقل حدود یک سال Bid/Ask واقعی و پوشش روزهای کاری بررسی می‌شود؛ داده کوتاه به‌عنوان آزمون بلند پذیرفته نمی‌شود. گزارش تغییر اکوییتی OOS سالانه/فصلی جدا ذخیره می‌شود. این هنوز یک split است، نه اثبات پایداری یا چند fold.

بررسی تکمیلی انجام‌شده روی ۳۷۵٬۱۹۴ کندل M1 سال۲۰۲۱ از آینه عمومی EURUSD فقط ساختاری بود: شروع از۱۱ژانویه، شش تاریخ کاری غایب، حدود۱۲٫۵٪ کندل تخت، ساعت منبع نامشخص و بدون Bid/Ask. baseline سخت‌گیرانه هیچ تریگر نداد؛ نسخه کلوز دو تریگر داد. بنابراین ادعای بازده، کاهش استاپ یا موفقیت مدل از این تاریخچه نمی‌کنیم. گزارش در reports/pivot_year است.

## پروتکل آزمایش و مدل کاهش خطا

چهار نسخه **همین منطق**: سایه/کلوز × TP۲٫۵/۳R. baseline از پیش `wick_2.5R` است؛ بهترین نتیجه تست انتخاب نمی‌شود. واگرایی زمانی فقط روی development به‌صورت تشخیصی گزارش می‌شود. تقسیم زمانی ۶۰٪ آموزش، ۲۰٪ اعتبارسنجی، ۲۰٪ آزمون؛ این اجرای اولیه exploratory است، نه ارزیابی چند fold. برای تأیید نهایی چندساله، بازار دیگر و >=۲۴ ماه OOS لازم است.

ML برچسب معاملات مستقل تیک‌به‌تیک همین ستاپ را می‌خواند؛ فقط ویژگی‌های قابل مشاهده در تریگر، نه R تصویر یا آینده. آموزش با exposure غیرهم‌پوشان و embargo۲۴ساعت؛ حداقل ۱۲ماه ورودی، ۱۰۰ label آموزش، ۳۰ برد و ۳۰ باخت اعتبارسنجی. انتخاب آستانه با حفظ >=۹۵٪ تعداد و پول بردها، حذف >=۱۰٪ باخت‌ها و PnL بهتر. **این قید تضمین حفظ برد آینده نیست.**

آستانه و ضرایب قبل از بک‌تست نهایی در `frozen_before_test.json` قفل می‌شوند. در نمونه کوتاه ML bypass است؛ غیرفعال بودن به‌معنای بهترشدن سیستم نیست. گزارش paired می‌گوید چند برد/باخت و چه مقدار سود حذف شده؛ گزارش حساب مشترک جداست چون حذف یک معامله ممکن است معامله بعدی را ممکن کند.

سلول آخر: گزارش تمامی نسخه‌ها، تنش هزینه، funnel ابطال‌ها، ردگیری A/B/C/D و علت‌ها، اکوییتی هر تیک، معاملات و R خالص، ماه‌های کامل/ناقص، نمودار ستاپ‌های **واقعی GBPUSD** و گزارش HTML. مسیر `artifacts/pivot`. فایل equity بزرگ در ZIP گزارش قرار نمی‌گیرد.


In [ ]:
# @title ۵ — اجرای پژوهش، بک‌تست واقعی و گزارش جامع
mode='sample' if DATA_MODE=='sample' else 'research'
if mode=='sample':
    report,comparison,results=run_pivot_experiment(ticks,data_audit,execution,
                                                 folder='artifacts/pivot',mode=mode,bars=bars)
else:
    # Enforce >=one calendar year of genuine quotes and inspect missing weekday coverage.
    report,comparison,results=run_long_pivot(ticks,data_audit,execution,
                                           folder='artifacts/pivot',minimum_days=365)
Path('artifacts/pivot/image_review.csv').write_text(image_annotations.to_csv(index=False))
print(json.dumps(report,indent=2,ensure_ascii=False,default=str))
display(comparison)
display(HTML(Path('artifacts/pivot/report.html').read_text()))
import zipfile
with zipfile.ZipFile('HIPO_Pivot_Report.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('artifacts/pivot').glob('*'):
        if p.is_file() and p.suffix in ['.csv','.json','.html','.png']:z.write(p,arcname=p.name)
print('Report ZIP: HIPO_Pivot_Report.zip')
print('Every-tick equity: artifacts/pivot/test_tick_equity_*.parquet')
if 'google.colab' in sys.modules:
    from google.colab import files
    print('Download with: files.download("HIPO_Pivot_Report.zip")')
